[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/https://github.com/maurergroup/MLinCP/Module_XI_Generative_Models/WS9_Generative_Models/WS5_Generative_Models.ipynb)

# WS 9 — Generative Models

**Machine Learning in Computational Physics** · University of Vienna

---

<div class="alert alert-block alert-danger">

**Assignment submission**

When you submit this notebook for assignment, make sure that all tasks below (blue boxes) are completed (with code) and all questions are answered (with extra markdown responses). Make sure that the notebook runs correctly when all cells are executed in the right order from top to bottom. Before submission, do not clear outputs. Leave all the outputs from the last run included.
</div>

## 0. Prelude

In this workshop, we will tackle the problem of generating configurations in the canonical ensemble using **normalizing flows**. Our goal is to train an invertible neural network that transforms configurations sampled from a simple reference distribution into configurations distributed according to the Boltzmann distribution of a crystalline Lennard-Jones system.

As a reference distribution, we will use an **Einstein crystal**, in which particles fluctuate independently around the sites of a perfect lattice. Samples from this model are straightforward to generate, since the particle displacements are Gaussian distributed. The target distribution, on the other hand, is the Boltzmann distribution associated with a Lennard-Jones solid, which exhibits non-trivial correlations between particle positions.

The central idea is to learn a transformation that maps configurations from the Einstein crystal onto configurations representative of the Lennard-Jones crystal. Once trained, the flow can be used both to generate equilibrium configurations and to estimate thermodynamic quantities through importance sampling.

This workshop is inspired by the work of Wirnsberger *et al.* [Wirnsberger2022], who demonstrated that normalizing flows can be used to efficiently sample molecular systems and compute free-energy differences. Interested readers are encouraged to consult the original reference for a more comprehensive discussion of the methodology and its applications.

Throughout the notebook, we will gradually build the components of a normalizing flow model, train it on a Lennard-Jones crystal, and analyze its performance using tools from statistical mechanics and machine learning.

The workshop relies primarily on the **PyTorch** library. In addition, a small collection of helper modules is provided to avoid spending time on lengthy implementations that are not central to the learning objectives of the notebook. These modules contain either standard algorithms, utility functions, or code whose implementation would be too cumbersome for the scope of the workshop.

The most important files are listed below. Participants are encouraged to inspect the source code, docstrings, and comments for additional details.

### `systems.py`

This file contains the classes defining the **prior** and **posterior** probability distributions used throughout the workshop. Each system provides methods to

- compute the potential energy through `energy`,
- generate an initial configuration through `init_conf`,
- sample configurations through `sample`.

In particular, the file contains the classes

- `EinsteinCrystal3D`, representing the harmonic reference system,
- `LennardJones3D`, representing the target Lennard-Jones crystal.

### `samplers.py`

This file contains the `MetropolisMonteCarlo` class, which implements Metropolis Monte Carlo sampling for a system of particles at fixed volume and temperature.

The main routine is `sample_space`, which can be used to generate equilibrium configurations from a given system and will be employed to generate reference data for training and validation.

### `utils.py`

This file collects several utility functions used throughout the notebook, including

- the computation of the radial distribution function,
- the implementation of rational quadratic spline transformations,
- helper routines for numerically stable mathematical operations,
- helper functions used to construct coupling-layer coordinate partitions.

The rational quadratic spline implementation and associated helper functions are adapted from the `nflows` package developed by Bayesians AI:

https://github.com/bayesiains/nflows

These spline transformations constitute the main building block of the normalizing flow model developed in this workshop.

In [ ]:
from pathlib import Path
import sys

module_path = Path.cwd()

if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print(f"Module path: {module_path}")

import os
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import random_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from einops import rearrange
from itertools import combinations

from systems import EinsteinCrystal3D, LennardJones3D
from samplers import MetropolisMonteCarlo
from utils import radial_distribution_function, rational_quadratic_spline, get_target_indices

# Locate repo root (contains pyproject.toml) — works in Docker, VS Code, and local Jupyter
_p = Path().resolve()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
DATA_DIR = _p.joinpath('data')
# DATA_DIR = "./"

In [ ]:
# ============================================================
# Device selection
# ============================================================

# CUDA: NVIDIA GPUs
if torch.cuda.is_available():

    device = torch.device("cuda")

    print("Using CUDA GPU")
    print(torch.cuda.get_device_name(0))

# MPS: Apple Silicon GPU (MacOS)
elif torch.backends.mps.is_available():

    device = torch.device("mps")

    print("Using Apple Silicon GPU (MPS)")

# CPU fallback
else:

    device = torch.device("cpu")

    print("Using CPU")

## 1. Data Preparation and Analysis

#### The Lennard-Jones Crystal

The system studied in this workshop is a crystal of Lennard-Jones particles in reduced units. The pair interaction is described by the Lennard-Jones potential

$$
U_{\mathrm{LJ}}(r)
=
4
\left[
\left(\frac{1}{r}\right)^{12}
-
\left(\frac{1}{r}\right)^6
\right].
$$

The thermodynamic state point considered throughout the notebook is characterized by

$$
N = 108,
\qquad
\rho^* = 1.1,
\qquad
T^* = 1.0,
$$

which corresponds to a crystalline solid. Figure 1 shows the phase diagram of the Lennard-Jones system and indicates that the chosen state point lies well within the solid FCC region.
<p align="center">
<a href="https://commons.wikimedia.org/wiki/File:LJ_PhaseDiagram.png#/media/File:LJ_PhaseDiagram.png">
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/56/LJ_PhaseDiagram.png/1280px-LJ_PhaseDiagram.png" alt="LJ PhaseDiagram.png" width="500">
</a>
<br>
By <a href="https://en.wikipedia.org/wiki/User:TimeStep89" class="extiw" title="w:User:TimeStep89">TimeStep89</a> - I created the plot, <a href="https://creativecommons.org/licenses/by/4.0" title="Creative Commons Attribution 4.0">CC BY 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=133325553">Link</a></p>

<p align="center">
<b>Figure 1.</b> Phase diagram of the Lennard-Jones system.
</p>

To reduce the computational cost, the potential is truncated and shifted at

$$
r_{\mathrm{cut}}
=
\min\!\left(
2.5,
\frac{L}{2}
\right),
$$

so that both the potential energy and the forces vanish smoothly at the cutoff distance. In addition, the repulsive core is linearized below

$$
r_{\mathrm{in}} = 0.8,
$$

to prevent numerical instabilities arising from accidental particle superposition.

The target probability distribution that we aim to sample is the canonical Boltzmann distribution,

$$
p(\mathbf{x})
=
\frac{
e^{-\beta U(\mathbf{x})}
}{
Z
},
$$

where $U(\mathbf{x})$ denotes the total Lennard-Jones potential energy of the configuration $\mathbf{x}$, $\beta = 1/T^*$ is the inverse temperature in reduced units, and $Z$ is the configurational partition function.

### The Einstein Crystal Prior

As a reference distribution, we employ the **Einstein crystal**, a system of independent particles fluctuating around the sites of a perfect lattice. Its energy is given by

$$
U_{\mathrm{EC}}(\mathbf{x})
=
\frac{1}{2\sigma^2}
\sum_{i=1}^{N}
\left|
\mathbf{x}_i
-
\mathbf{x}_i^{\mathrm{ref}}
\right|^2,
$$

where $\mathbf{x}_i^{\mathrm{ref}}$ denotes the position of the $i$-th lattice site and $\sigma$ controls the amplitude of the fluctuations around the reference crystal.

The corresponding probability distribution is a product of independent Gaussian distributions centered at the lattice sites. To ensure that each particle remains associated with its reference site, the Gaussian fluctuations are truncated at a distance $r_{\mathrm{cut}}$ from the corresponding lattice position.

In [ ]:
n_particles = 108
dimensions = 3

temperature = 1.0
beta = 1.0 / temperature

rho = 1.1

prior = EinsteinCrystal3D(
    n_particles=n_particles,
    rho=rho,
    device=device,
    displacement_std=0.07,
    truncation_radius=1.5,
)

system = LennardJones3D(
    n_particles=n_particles,
    rho=rho,
    device=device,
    linearization_radius=0.8,
)

print(
    f"rho = {rho:.4f}, "
    f"T = {temperature:.4f}, "
    f"L = {system.box_size:.4f}"
)

Before constructing and training the normalizing flow, we generate a set of reference configurations that will be used throughout the workshop for validation, visualization, and, in some cases, training.

Sampling configurations from the Einstein crystal is straightforward, since particles fluctuate independently around their lattice sites. As a result, Einstein crystal configurations are generated on the fly whenever they are needed.

Generating equilibrium configurations of the Lennard-Jones crystal is more computationally demanding. To avoid spending a significant amount of time running Monte Carlo simulations during the workshop, a set of precomputed configurations is provided. These configurations were generated using Metropolis Monte Carlo sampling and are automatically loaded when the notebook is executed.

Reference configurations are stored in a `DATA_DIR` directory specified by the user. Whenever the notebook is executed, it first checks whether a suitable cache file corresponding to the chosen thermodynamic state point is already available. If such a file is found, the configurations are loaded directly from disk. Otherwise, a new Monte Carlo simulation is performed, and the resulting configurations and energies are saved to the directory for future use.

This mechanism allows expensive simulations to be reused across multiple runs while ensuring that changing the system parameters automatically triggers the generation of new reference data.

When running Monte Carlo simulations, it is important to monitor the acceptance ratio. As a rule of thumb, the acceptance probability should lie between approximately 30% and 60%. If the acceptance ratio falls significantly outside this range, adjust the maximum displacement used in the Monte Carlo moves accordingly.

In [ ]:
reference_samples_prior, reference_energy_prior = prior.sample(n_samples=8192, return_energy=True)
identity_energy_prior = system.energy(reference_samples_prior)

In [ ]:
# --------------------------------------------------
# Cache directory
# --------------------------------------------------

cache_dir = os.path.join(DATA_DIR, "LJ/N108/")
print(cache_dir)
# os.makedirs(
#     cache_dir,
#     exist_ok=True,
# )

In [ ]:
# --------------------------------------------------
# Metropolis Monte Carlo sampling
# --------------------------------------------------

mc_sampler = MetropolisMonteCarlo(
    system=system,
    step_size=0.05,
    n_cycles=100,
    n_equilibration=1000,
)

# ==================================================
# Reference posterior samples
# ==================================================

cache_file = os.path.join(
    cache_dir,
    (
        f"lj_"
        f"N{system.n_particles}_"
        f"rho{system.rho:.4f}_"
        f"T{temperature:.4f}.pt"
    ),
)

if os.path.exists(cache_file):

    print(f"Loading reference data from\n"
          f"    {cache_file}"
    )

    data = torch.load(cache_file, map_location=device)

    reference_samples_system = data["samples"]
    reference_energy_system = data["energy"]

else:

    print("Generating reference LJ samples...")

    reference_samples_system, reference_energy_system = (
        system.sample(
            n_samples=8192,
            beta=beta,
            sampler=mc_sampler,
            return_energy=True,
        )
    )

    torch.save(
        {
            "samples":reference_samples_system,
            "energy":reference_energy_system,
            "beta":beta,
            "temperature":temperature,
            "rho":system.rho,
            "n_particles":system.n_particles,
        },
        cache_file,
    )

    print(f"Saved reference data to\n"
          f"    {cache_file}"
    )

print()
print(
    f"Samples shape : "
    f"{reference_samples_system.shape}"
)

print(
    f"Energy shape  : "
    f"{reference_energy_system.shape}"
)

Before training the normalizing flow, it is useful to compare the prior and posterior distributions. Two particularly informative quantities are the **potential energy distribution** and the **radial distribution function** $g(r)$.

The potential energy distribution provides information about the regions of configuration space explored by the system. Since the Boltzmann distribution is given by

$$
p(\mathbf{x})
=
\frac{e^{-\beta U(\mathbf{x})}}{Z},
$$

the energy can be interpreted, up to an additive constant, as

$$
U(\mathbf{x})
=
-k_{\mathrm{B}}T\,\log p(\mathbf{x}).
$$

Configurations with low energy therefore correspond to highly probable regions of configuration space.

When analyzing samples from the prior distribution, however, it is important to distinguish between the **prior energy** and the **posterior energy**. Since our objective is to generate samples distributed according to the Lennard-Jones Boltzmann distribution, evaluating prior samples using the Einstein crystal energy is often not very informative. Instead, it is much more useful to evaluate prior samples using the Lennard-Jones energy.

More precisely, if

$$
z \sim \mu_Z(z)
$$

denotes a sample from the Einstein crystal and

$$
x = F(z)
$$

denotes the transformation learned by the normalizing flow, then evaluating the Lennard-Jones energy of the prior sample,

$$
U_{\mathrm{LJ}}(z),
$$

can be interpreted as evaluating

$$
U_{\mathrm{LJ}}(F(z))
$$

under the special choice

$$
F(z) = z.
$$

In other words, it measures the quality of the **identity transformation** as a transport map from the prior to the target distribution. Comparing this baseline to the energies obtained after applying the trained flow provides a direct indication of how much the learned transformation improves the overlap between the prior and posterior distributions.

The second quantity we will examine is the radial distribution function $g(r)$, which characterizes the local structure of the system. While the energy distribution measures how well the flow reproduces the correct probability density, the radial distribution function provides a physically intuitive measure of whether the generated configurations exhibit the correct crystalline order.

Both quantities will be used extensively throughout the notebook to assess the quality of the learned transport map and to monitor the progress of training.

In [ ]:
# --------------------------------------------------
# RDFs
# --------------------------------------------------

r_prior, g_prior = radial_distribution_function(
    configurations=reference_samples_prior,
    system=prior,
)

r_system, g_system = radial_distribution_function(
    configurations=reference_samples_system,
    system=system,
)

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4),
)

# --------------------------------------------------
# Energy distributions
# --------------------------------------------------

axes[0].hist(
    reference_energy_system.cpu().numpy(),
    bins=50,
    density=True,
    alpha=0.6,
    label="Reference: LJ Energy evaluated on LJ samples",
)

axes[0].hist(
    identity_energy_prior.cpu().numpy(),
    bins=50,
    density=True,
    alpha=0.6,
    label="Identity: LJ Energy evaluated on EC samples",
)

axes[0].set_xlabel("Energy")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Energy distributions")
axes[0].legend()

# --------------------------------------------------
# Radial distribution functions
# --------------------------------------------------

axes[1].plot(
    r_system,
    g_system,
    label="Lennard-Jones crystal",
)

axes[1].plot(
    r_prior,
    g_prior,
    label="Einstein crystal",
)

axes[1].set_xlabel(r"$r$")
axes[1].set_ylabel(r"$g(r)$")
axes[1].set_title("Radial distribution function")
axes[1].legend()

plt.tight_layout()
plt.show()

In addition to quantitative observables such as the energy distribution and the radial distribution function, it is often useful to visualize the configurations themselves.

Since both the Einstein crystal and the Lennard-Jones crystal fluctuate around the same underlying lattice structure, plotting particle positions relative to their lattice sites provides an intuitive picture of the differences between the two distributions. In particular, it allows us to directly observe the amplitude and shape of the thermal fluctuations around the equilibrium positions.

The next cell generates a visualization of the particle displacements from the reference lattice for configurations sampled from both the prior and posterior distributions. Comparing these plots helps build intuition about the transport problem that the normalizing flow must solve: the prior already captures the overall crystalline structure of the system, but it does not reproduce the detailed correlations and fluctuation patterns present in the Lennard-Jones crystal.

This observation is one of the main motivations for using an Einstein crystal as the prior distribution. Since the prior already resembles the target distribution, the normalizing flow only needs to learn a relatively small correction rather than constructing the entire crystalline structure from scratch.

In [ ]:
# --------------------------------------------------
# Reference FCC lattice
# --------------------------------------------------

reference_positions = system.init_conf(as_numpy=True)
reference_positions_batch = (reference_positions[None])

# --------------------------------------------------
# Utility
# --------------------------------------------------

def crystal_cloud(samples, system, n_display):

    indices = np.random.choice(
        samples.shape[0],
        size=min(n_display, samples.shape[0]),
        replace=False,
    )

    positions = (samples[indices].view(-1, n_particles, 3).cpu().numpy())

    # Minimum-image displacements relative to FCC sites

    displacements = (positions - reference_positions_batch)
    displacements -= (system.box_size * np.round(displacements / system.box_size))

    positions = (reference_positions_batch + displacements)

    return positions.reshape(-1, 3)

# --------------------------------------------------
# Build clouds
# --------------------------------------------------

n_display = 500

prior_cloud = crystal_cloud(
    reference_samples_prior,
    prior,
    n_display,
)

posterior_cloud = crystal_cloud(
    reference_samples_system,
    system,
    n_display,
)

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig = plt.figure(figsize=(8, 8))

ax = fig.add_subplot(111, projection="3d")

# Prior cloud

prior_handle = ax.scatter(
    prior_cloud[:,0],
    prior_cloud[:,1],
    prior_cloud[:,2],
    s=2,
    alpha=0.03,
)

# Posterior cloud

posterior_handle = ax.scatter(
    posterior_cloud[:,0],
    posterior_cloud[:,1],
    posterior_cloud[:,2],
    s=2,
    alpha=0.03,
)

# FCC lattice sites

fcc_handle = ax.scatter(
    reference_positions[:,0],
    reference_positions[:,1],
    reference_positions[:,2],
    s=10,
    c="k",
    marker="x",
)

legend = ax.legend(
    [prior_handle, posterior_handle, fcc_handle],
    ["Einstein crystal", "Lennard-Jones crystal", "FCC sites"],
)

for h in legend.legend_handles:
    h.set_alpha(1.0)
    h.set_sizes([40])

half_box = 0.5 * system.box_size

ax.set_xlim(-half_box, half_box)
ax.set_ylim(-half_box, half_box)
ax.set_zlim(-half_box, half_box)

ticks = [-half_box, 0.0, half_box]

ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_zticks(ticks)

ax.set_xticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_yticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_zticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax.set_box_aspect((1, 1, 1))

ax.set_title(
    f"{n_display} configurations"
)

# plt.tight_layout()
plt.show()

## 2. Flow Architecture

### 2A) A General Purpose Flow Assembler

The architecture used throughout this workshop is illustrated in the diagram below. At its core lies a sequence of **invertible coupling blocks**, which together define the normalizing flow. These blocks are responsible for learning the transformation between the prior and posterior distributions while preserving exact invertibility and providing access to the Jacobian determinant required for probability calculations.

In principle, the coupling blocks could operate directly on the physical coordinates of the particles. In practice, however, learning can often be simplified by transforming the configurations into a more convenient representation before they are processed by the flow. Examples include expressing particle positions as displacements from a reference lattice, normalizing coordinates to a standard range, or removing redundant degrees of freedom associated with symmetries of the system.

To accommodate these operations, the architecture separates the flow into two components:

- **Transformation layers**, which convert between physical coordinates and a representation that is more convenient for the neural network.
- **Invertible flow blocks**, which learn the actual transport map between the prior and posterior distributions.

The transformation layers are problem-dependent and can be added, removed, or modified without changing the structure of the flow itself. The coupling blocks, on the other hand, constitute the trainable core of the model and are responsible for constructing the invertible mapping between the two probability distributions.

The same architecture naturally supports both the **generative direction** (\(z \rightarrow x\)) and the **encoding direction** (\(x \rightarrow z\)). Since every component is exactly invertible, the inverse transformation is obtained simply by traversing the architecture in the opposite direction.

```text
      GENERATIVE DIRECTION (z → x)                 ENCODING DIRECTION (x → z)

            Prior coordinates                         Posterior coordinates
                    │                                          │
                    ▼                                          ▼
    ┌──────────────────────────────┐        ┌──────────────────────────────┐
    │ Prior-side transformations   │        │ Posterior-side              │
    │                              │        │ transformations             │
    │ • Lattice displacement       │        │ • Lattice displacement      │
    │ • Normalization              │        │ • Normalization             │
    │ • Symmetry removal           │        │ • Symmetry removal          │
    └──────────────────────────────┘        └──────────────────────────────┘
                    │                                          │
                    ▼                                          ▼
           Network coordinates                     Network coordinates
                    │                                          │
                    ▼                                          ▼
    ┌──────────────────────────────┐        ┌──────────────────────────────┐
    │      Invertible flow         │        │      Invertible flow         │
    │                              │        │                              │
    │  Coupling blocks + shifts    │        │  Coupling blocks + shifts    │
    │       (forward pass)         │        │       (inverse pass)         │
    └──────────────────────────────┘        └──────────────────────────────┘
                    │                                          │
                    ▼                                          ▼
           Network coordinates                     Network coordinates
                    │                                          │
                    ▼                                          ▼
    ┌──────────────────────────────┐        ┌──────────────────────────────┐
    │ Inverse posterior-side       │        │ Inverse prior-side           │
    │ transformations              │        │ transformations              │
    │                              │        │                              │
    │ • Denormalization            │        │ • Denormalization            │
    │ • Symmetry restoration       │        │ • Symmetry restoration       │
    │ • Coordinate reconstruction  │        │ • Coordinate reconstruction  │
    └──────────────────────────────┘        └──────────────────────────────┘
                    │                                          │
                    ▼                                          ▼
         Posterior coordinates                     Prior coordinates
```

In addition to implementing the forward and inverse transformations, the normalizing flow class provides the loss functions used during training. In this workshop we consider two complementary objectives, corresponding to the two directions of the flow.

#### Reverse KL (Training by Example)

The encoding direction maps configurations sampled from the target distribution,

$$
x \sim \mu_X(x),
$$

to latent configurations

$$
z = F^{-1}(x).
$$

Using the change-of-variables formula, the probability assigned by the flow to a configuration $x$ is

$$
p_X(x)
=
\mu_Z(z)
\left|
\det
J^{-1}(z)
\right|,
$$

where $\mu_Z(z)$ denotes the prior distribution.

Training by maximum likelihood is equivalent to minimizing the reverse Kullback-Leibler divergence,

$$
D_{\mathrm{KL}}
\!\left(
\mu_X \,\|\, p_X
\right),
$$

which leads to the loss

$$
\mathcal{L}_{xz}
=
-\frac{1}{N}
\sum_i
\log p_X(x_i).
$$

Using the change-of-variables formula, this can be written as

$$
\mathcal{L}_{xz}
=
-\frac{1}{N}
\sum_i
\left[
\log \mu_z(F(x_i))
+
\log
\left|
\det
J^{-1}(F(x_i))
\right|
\right].
$$

### Forward KL (Training by Energy)

The generative direction maps samples from the prior,

$$
z \sim \mu_Z(z),
$$

to physical configurations

$$
x = F(z).
$$

The corresponding objective is the forward Kullback-Leibler divergence,

$$
D_{\mathrm{KL}}
\!\left(
\mu_Z \,\|\, p_Z
\right),
$$

which, up to an additive constant, yields the loss

$$
\mathcal{L}_{zx}
=
\frac{1}{N}
\sum_i
\left[
\beta U(F(z_i))
-
\log
\left|
\det
J(F(z_i))
\right|
\right].
$$

Unlike maximum likelihood, this loss does not require reference configurations and can therefore be evaluated using only samples generated by the flow.

### Importance Weights

When evaluating the quality of a trained model, it is often useful to compute importance weights. These quantities are not required for training and can be expensive to evaluate, so they are typically computed only during validation or testing.

For samples generated by the flow,

$$
z \sim \mu_Z(z),
\qquad
x = F(z),
$$

the forward-direction importance weights are

$$
w_{zx}
=
\frac{
\mu_X(x)
}{
p_X(x)
}
=
\frac{
e^{-\beta U(F(z))}
}{
\mu_Z(z)
}
\left|
\det
J(F(z))
\right|.
$$

Taking the logarithm gives

$$
\log w_{zx}
=
-\beta U(F(z))
-
\log \mu_Z(z)
+
\log
\left|
\det
J(F(z))
\right|.
$$

Similarly, for configurations sampled from the target distribution,

$$
x \sim \mu_X(x),
\qquad
z = F^{-1}(x),
$$

the reverse-direction weights are

$$
w_{xz}
=
\frac{
\mu_Z(z)
}{
p_Z(z)
}
=
\mu_Z(F^{-1}(x))
\left|
\det
J^{-1}(F^{-1}(x))
\right|
e^{\beta U(x)},
$$

with logarithm

$$
\log w_{xz}
=
\log \mu_Z(F^{-1}(x))
+
\log
\left|
\det
J^{-1}(x)
\right|
+
\beta U(x).
$$

These weights can be used to estimate observables by reweighting and to compute diagnostics such as the relative effective sample size (rESS), which provides a quantitative measure of the overlap between the learned distribution and the target distribution.

### Implement the Flow Loss Functions

<div class="alert alert-block alert-info">

**TASK**

1. Inspect the methods `forward_kl_loss` and `backward_kl_loss` of the `NormalizingFlow` class.

2. Using the equations introduced in the previous section, implement the forward and reverse KL objectives.

3. Make sure that the correct change-of-variables terms are included through the logarithm of the Jacobian determinant.

4. Verify that both functions return a scalar loss that can be minimized during training.

5. Optionally, compute the logarithmic importance weights associated with each direction. These weights will later be used to evaluate the relative effective sample size (rESS) of the flow.

</div>

> **Note**
>
> Throughout this workshop, energies, log-probabilities, and logarithms of Jacobian determinants are represented as tensors of shape
>
> ```python
> [batch_size, 1]
> ```
>
> rather than
>
> ```python
> [batch_size]
> ```
>
> This convention originates from previous implementations and is maintained here for consistency. When implementing the loss functions, pay attention to tensor shapes. In particular, energy terms, log-probabilities, and Jacobian contributions should all have compatible shapes before being combined.

In [ ]:
class NormalizingFlow(nn.Module):
    """
    Normalizing flow mapping a prior onto
    a posterior.

    The transformation is organized into three stages:

        prior coordinates
                ↓
        prior-side transformations
                ↓
        network coordinates
                ↓
        invertible flow blocks
                ↓
        network coordinates
                ↓
        inverse posterior-side transformations
                ↓
        posterior coordinates

    The flow itself operates exclusively in network
    coordinates. The prior-side and posterior-side
    transformations are used to convert between physical
    coordinates and the representation used by the flow.

    Examples of such transformations include

        - expressing configurations as lattice
        displacements,
        - normalization to a standard domain,
        - removal of translational degrees of freedom,
        - removal and reinsertion of a reference particle.

    Such transformations allow the flow to operate on
    a more compact and physically meaningful coordinate
    system, reducing the burden on the neural network
    and improving training efficiency.

    Each flow block must implement

        block(x, inverse=False)

    and return

        transformed_x,
        logdetJ

    where logdetJ is the logarithm of the Jacobian
    determinant of the transformation.

    Likewise, each coordinate transformation layer must
    implement the same interface

        transform(x, inverse=False)

    allowing transformations and flow blocks to share a
    common API.

    Parameters
    ----------
    prior : EinsteinCrystal3D
        Prior distribution.

    posterior : LennardJones3D
        Target distribution.

    blocks : list[nn.Module]
        Sequence of invertible flow transformations acting
        in network coordinates.

    prior_sided_transformations : list[nn.Module], optional
        Transformations applied to prior samples before
        entering the flow.

        These map

            prior coordinates
                ->
            network coordinates.

    posterior_sided_transformations : list[nn.Module], optional
        Transformations applied to posterior samples before
        entering the inverse flow.

        These map

            posterior coordinates
                ->
            network coordinates.

    device : torch.device, optional
        Device used for computations.

    Notes
    -----
    If

        T_prior

    denotes the composition of prior-side transformations,

        F

    the normalizing flow, and

        T_post

    the composition of posterior-side transformations,

    then the generative map implemented by this class is

        x = T_post^{-1}
            ∘ F
            ∘ T_prior(z)

    while the inverse map is

        z = T_prior^{-1}
            ∘ F^{-1}
            ∘ T_post(x).

    All Jacobian contributions from coordinate
    transformations and flow blocks are accumulated
    automatically.
    """
    def __init__(
        self,
        prior,
        posterior,
        blocks,
        prior_sided_transformations=None,
        posterior_sided_transformations=None,
        device=None,
    ):

        super().__init__()

        self.prior = prior
        self.posterior = posterior

        if device is None:

            if torch.cuda.is_available():
                self.device = torch.device("cuda")
            elif (
                torch.backends.mps.is_available()
                and torch.backends.mps.is_built()
            ):
                self.device = torch.device("mps")
            else:
                self.device = torch.device("cpu")
        else:
            self.device = device

        if prior_sided_transformations is None:
            prior_sided_transformations = []

        if posterior_sided_transformations is None:
            posterior_sided_transformations = []

        self.prior_sided_transformations = nn.ModuleList(prior_sided_transformations)
        self.posterior_sided_transformations = nn.ModuleList(posterior_sided_transformations)

        self.blocks = nn.ModuleList(blocks)

        self.apply(self.init_weights)

    def init_weights(self, module):
        """
        Initialize coupling networks close to the identity.

        All linear layers are initialized with zero weights
        and biases. For spline coupling layers this produces
        an approximately identity transformation at the
        beginning of training.
        """

        if isinstance(module, nn.Linear):

            module.weight.data.fill_(0.0)
            # nn.init.normal_(module.weight, mean=0.0, std=1e-2)

            if module.bias is not None:

                module.bias.data.fill_(0.0)
                # nn.init.normal_(module.bias, mean=0.0, std=1e-2)


    def forward(self, z):
        """
        Apply the flow in the generative direction

            z -> x.

        The transformation is composed of three stages:

        1. Prior-side coordinate transformations

            prior coordinates
                    ->
            network coordinates

        2. Invertible flow blocks operating in network
        coordinates.

        3. Inverse posterior-side coordinate transformations

            network coordinates
                    ->
            posterior coordinates

        Parameters
        ----------
        z : torch.Tensor
            Prior configurations with shape

                [batch_size, dofs]

            expressed in the physical coordinates of the
            prior distribution.

        Returns
        -------
        x : torch.Tensor
            Generated configurations with shape

                [batch_size, dofs]

            expressed in the physical coordinates of the
            posterior distribution.

        logdetJ_zx : torch.Tensor
            Logarithm of the Jacobian determinant of the
            complete transformation

                z -> x

            with shape

                [batch_size, 1].

        Notes
        -----
        If

            T_prior

        denotes the prior-side coordinate transformation,

            F

        the normalizing flow, and

            T_post

        the posterior-side coordinate transformation,

        then this method evaluates

            x = T_post^{-1}
                ∘ F
                ∘ T_prior (z)

        while accumulating the corresponding Jacobian
        contributions.
        """
        x = z.clone()
        logdetJ_zx = z.new_zeros(z.shape[0], 1)

        # Prior -> network coordinates

        for transform in self.prior_sided_transformations:

            x, partial_logdetJ = transform(x, inverse=False)
            logdetJ_zx += partial_logdetJ

        # Flow

        for block in self.blocks:

            x, partial_logdetJ = block(x, inverse=False)
            logdetJ_zx += partial_logdetJ

        # Network -> posterior coordinates

        for transform in reversed(self.posterior_sided_transformations):

            x, partial_logdetJ = transform(x, inverse=True)
            logdetJ_zx += partial_logdetJ

        return x, logdetJ_zx

    def inverse(self, x):
        """
        Apply the flow in the inverse direction

            x -> z.

        The transformation is composed of three stages:

        1. Posterior-side coordinate transformations

            posterior coordinates
                    ->
            network coordinates

        2. Inverse flow blocks operating in network
        coordinates.

        3. Inverse prior-side coordinate transformations

            network coordinates
                    ->
            prior coordinates

        Parameters
        ----------
        x : torch.Tensor
            Configurations with shape

                [batch_size, dofs]

            expressed in the physical coordinates of the
            posterior distribution.

        Returns
        -------
        z : torch.Tensor
            Mapped configurations with shape

                [batch_size, dofs]

            expressed in the physical coordinates of the
            prior distribution.

        logdetJ_xz : torch.Tensor
            Logarithm of the Jacobian determinant of the
            complete transformation

                x -> z

            with shape

                [batch_size, 1].

        Notes
        -----
        If

            T_prior

        denotes the prior-side coordinate transformation,

            F

        the normalizing flow, and

            T_post

        the posterior-side coordinate transformation,

        then this method evaluates

            z = T_prior^{-1}
                ∘ F^{-1}
                ∘ T_post (x)

        while accumulating the corresponding Jacobian
        contributions.
        """
        z = x.clone()
        logdetJ_xz = x.new_zeros(x.shape[0],1)

        # Posterior -> network coordinates

        for transform in self.posterior_sided_transformations:

            z, partial_logdetJ = transform(z, inverse=False)
            logdetJ_xz += partial_logdetJ

        # Flow

        for block in reversed(self.blocks):

            z, partial_logdetJ = block(z, inverse=True)
            logdetJ_xz += partial_logdetJ

        # Network -> prior coordinates

        for transform in reversed(self.prior_sided_transformations):

            z, partial_logdetJ = transform(z, inverse=True)
            logdetJ_xz += partial_logdetJ

        return z, logdetJ_xz

    def forward_kl_loss(
        self,
        z,
        beta,
        energy_z=None,
    ):
        """
        Compute the forward KL objective

            KL(q_theta(x) || p(x))

        using samples drawn from the prior.

        Parameters
        ----------
        z : torch.Tensor
            Prior samples.

        beta : float
            Inverse temperature of the target system.

        energy_z : torch.Tensor, optional
            Precomputed prior energies.

        Returns
        -------
        loss : torch.Tensor
            Scalar training loss.

        logw_zx : torch.Tensor or None
            Importance weights used for evaluation.
            Returned only in evaluation mode.
        """

        x, logdetJ_zx = ... # Your code goes here
        logp_zx = ... # Your code goes here
        logw_zx = None

        if not self.training:

            if energy_z is None:
                energy_z = ...  # Your code goes here

            logp_z = ... # Your code goes here

            logw_zx = ...  # Your code goes here

        loss = ...  # Your code goes here
        
        return loss, logw_zx

    def reverse_kl_loss(
        self,
        x,
        beta,
        energy_x=None,
    ):
        """
        Compute the reverse KL objective

            KL(p(x) || q_theta(x))

        using samples drawn from the target distribution.

        Parameters
        ----------
        x : torch.Tensor
            Target samples.

        beta : float
            Inverse temperature of the target system.

        energy_x : torch.Tensor, optional
            Precomputed target energies.

        Returns
        -------
        loss : torch.Tensor
            Scalar training loss.

        logw_xz : torch.Tensor or None
            Importance weights used for evaluation.
            Returned only in evaluation mode.
        """

        z, logdetJ_xz =  ...
        logp_xz = ...  # Your code goes here
        logw_xz = None

        if not self.training:

            if energy_x is None:
                energy_x = ...  # Your code goes here

            logp_x = ...  # Your code goes here

            logw_xz = ...  # Your code goes here

        loss = ...  # Your code goes here
        
        return loss, logw_xz

### 2B) Predicting Spline Parameters

The coupling layers of the normalizing flows will be represented by rational quadratic splines whose parameters will be determined by a neural network. In our implementation, these parameters are predicted by a small **permutation-equivariant transformer network**.

The role of this network is to analyze the coordinates that remain unchanged by the coupling layer and use them to predict the parameters of the spline acting on the transformed coordinates. Depending on the coupling pattern, the network outputs the widths, heights, and derivatives that define the rational quadratic spline transformation.

The network is composed of three stages:

1. **Circular encoding**
2. **Transformer encoder**
3. **Linear output projection**

The overall architecture can be summarized as

```text
Input coordinates
        │
        ▼
Circular encoding
        │
        ▼
Linear embedding
        │
        ▼
Transformer encoder
        │
        ▼
Linear projection
        │
        ▼
Spline parameters
```

#### Circular Encoding

The input coordinates are assumed to lie in the interval

$$
[-1,1].
$$

Before being processed by the transformer, each coordinate is mapped to a periodic representation using a Fourier basis:

$$
x
\;\longrightarrow\;
\left(
\cos(\pi x),
\sin(\pi x),
\dots,
\cos(n_f\pi x),
\sin(n_f\pi x)
\right),
$$

where $n_f$ denotes the number of frequencies.

This encoding serves two purposes. First, it enriches the representation available to the neural network, allowing it to model highly non-linear dependencies. Second, it provides a natural way of representing periodic structures and smooth variations around lattice positions.

#### Transformer Encoder

After the Fourier encoding, the coordinates of each particle are projected into a higher-dimensional feature space and processed by a transformer encoder.

The transformer applies self-attention between particles, allowing each particle to exchange information with every other particle in the configuration. This is particularly useful in condensed matter systems, where the transformation applied to one particle generally depends on the arrangement of its neighbors.

An important property of the architecture is that it is **permutation equivariant**. If the particles are reordered at the input, the outputs are reordered in exactly the same way. This property ensures that the network respects the indistinguishability of particles and avoids introducing arbitrary ordering effects.

#### Output Layer

The final linear layer maps the transformer features to the parameters required by the coupling layer.

For a rational quadratic spline with $K$ bins, the network predicts $K$ bin widths, $K$ bin heights, and $K+1$ boundary derivatives (or an equivalent parameterization), for each transformed coordinate. These parameters are then used to construct an invertible spline transformation whose Jacobian determinant can be evaluated exactly.

The transformer therefore does not directly transform the coordinates. Instead, it acts as a parameter generator that adapts the spline transformation to the local environment of each particle.

In [ ]:
class ParameterEquivariantNetwork(nn.Module):
    """
    Permutation-equivariant transformer network used to
    predict coupling-layer parameters.

    The network acts independently on each particle while
    allowing information exchange through self-attention.

    A circular Fourier encoding is applied to each input
    coordinate before the transformer.

    Parameters
    ----------
    input_size : int
        Number of input features per particle.

    output_size : int
        Number of output features predicted per particle.

    transformer_args : dict, optional
        Transformer architecture parameters.

        Supported keys are

            dim
            depth
            n_heads

    n_freqs : int, default=8
        Number of Fourier frequencies used in the circular
        encoding.

    Notes
    -----
    Input coordinates are assumed to lie in the interval

        [-1, 1].

    Each coordinate x is encoded as

        cos(k π x),
        sin(k π x)

    for

        k = 1, ..., n_freqs.

    This encoding provides a periodic representation of
    particle positions and improves learning of periodic
    structures.
    """
    def __init__(self, input_size, output_size, transformer_args={"depth" : 1, "dim" : 128}, n_freqs=8):
        super().__init__()

        self.input_size = input_size
        self.output_size = output_size

        self.n_freqs = n_freqs
        self.register_buffer("freqs", torch.arange(n_freqs, dtype=torch.float32,).view(1, 1, -1) + 1)
        
        self.lin_in = nn.Linear(self.input_size * 2 * self.n_freqs, transformer_args["dim"])
        
        self.transformer_encoder = nn.TransformerEncoder(
                                        nn.TransformerEncoderLayer(
                                            d_model = transformer_args["dim"], 
                                            nhead = transformer_args["dim"]//64, 
                                            dim_feedforward = transformer_args["dim"]*4, 
                                            batch_first = True, 
                                            norm_first = True, 
                                            dropout=0.0
                                        ), 
                                        transformer_args["depth"]
                                    )
        
        self.lin_out = nn.Linear(transformer_args["dim"], self.output_size)

    def forward(self, x):
        """
        Predict coupling-layer parameters.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor with shape

                [batch_size, n_particles * input_size]

        Returns
        -------
        torch.Tensor
            Predicted parameters with shape

                [batch_size, n_particles * output_size]
        """
        batch_size = x.shape[0]

        x = x.reshape(batch_size, -1, self.input_size)

        # Circular encoder (input must be between -1 and 1)
        cos_enc = torch.cos(self.freqs * torch.pi * x.unsqueeze(-1))
        sin_enc = torch.sin(self.freqs * torch.pi * x.unsqueeze(-1))

        x = torch.cat([
                cos_enc.view(batch_size, -1, self.input_size * self.n_freqs),
                sin_enc.view(batch_size, -1, self.input_size * self.n_freqs)
            ], dim=-1)

        x = self.lin_in(x)
        x = self.transformer_encoder(x)
        x = self.lin_out(x)

        x = x.reshape(batch_size, -1)

        return x

### Rational Quadratic Spline Coupling Layer

The coupling layer is the fundamental building block of the normalizing flow. Its purpose is to apply a flexible, non-linear, and exactly invertible transformation to a subset of the coordinates while leaving the remaining coordinates unchanged.

Given an input configuration $x = (x_{\mathrm{id}}, x_{\mathrm{tr}})$ the coordinates are partitioned into two groups:

- **identity coordinates** $x_{\mathrm{id}}$, which are left unchanged,
- **transformed coordinates** $x_{\mathrm{tr}}$, which are mapped through an invertible spline transformation.

The identity coordinates act as a conditioning signal that determines how the transformed coordinates should be modified. This structure guarantees invertibility because the transformation applied to $x_{\mathrm{tr}}$ depends only on coordinates that remain unchanged.

The overall architecture of a coupling block is

```text
Input coordinates
        │
        ▼
Coordinate split
        │
        ├──────────────► Identity coordinates
        │                       │
        │                       ▼
        │            Equivariant transformer
        │                       │
        │                       ▼
        │              Spline parameters
        │
        ▼
Transformed coordinates
        │
        ▼
Rational quadratic spline
        │
        ▼
Output coordinates
```

#### Coordinate Partitioning

The first step consists of selecting which coordinate directions should be transformed. For example,

```python
target_coordinates = (0,)
```

transforms all $x$ coordinates while conditioning on the remaining directions, whereas

```python
target_coordinates = (0, 2)
```

transforms the $x$ and $z$ coordinates while conditioning on the $y$ coordinates.

The helper function `get_target_indices` constructs the corresponding index sets and splits the flattened coordinate vector into conditioned and transformed degrees of freedom.

#### Predicting Spline Parameters

The identity coordinates are passed to the permutation-equivariant transformer network described previously. The transformer predicts the parameters defining the spline transformation for each transformed coordinate.

For a rational quadratic spline with $K$ bins, the network predicts

- $K$ bin widths,
- $K$ bin heights,
- $K$ knot derivatives.

The final derivative required by the spline is not predicted independently. Instead, it is constrained to be equal to the first derivative,

$$
s_{K} = s_{0},
$$

which enforces periodicity of the transformation. Consequently, the network outputs a total of $3K$ parameters for each transformed degree of freedom.
The output of the network is therefore organized as

$$
[\text{widths} \;|\; \text{heights} \;|\; \text{slopes}],
$$

for every transformed coordinate.

#### Rational Quadratic Spline Transformation

Once the spline parameters have been predicted, the transformed coordinates are mapped through a monotonic rational quadratic spline.

Unlike affine coupling layers, which are restricted to scaling and shifting coordinates, spline couplings can represent highly non-linear monotonic transformations. This significantly increases the expressive power of the flow while preserving exact invertibility.

A typical spline transformation is illustrated schematically below:

```text
y
│
│        /
│      /
│    /
│  /
│/
└────────────── x
```

The actual transformation consists of multiple rational quadratic segments joined together in such a way that both the function and its derivative remain continuous.

#### Periodicity

The coordinates processed by the flow have been normalized to the interval

$$
[-1,1].
$$

To avoid introducing artificial discontinuities at the boundaries, the spline is made periodic by enforcing

$$
s_{\mathrm{first}}
=
s_{\mathrm{last}},
$$

where $s$ denotes the spline derivative at a knot.

This constraint ensures that the spline joins smoothly at the endpoints of the interval and is therefore well suited to periodic systems.

#### Jacobian Determinant

A key advantage of rational quadratic splines is that both the inverse transformation and the Jacobian determinant can be computed analytically.

The Jacobian of a coupling layer is triangular because the identity coordinates are left unchanged. Consequently, the logarithm of the determinant reduces to a sum over the transformed coordinates,

$$
\log
\left|
\det J
\right|
=
\sum_i
\log
\left|
\frac{\partial y_i}
{\partial x_i}
\right|.
$$

This quantity is returned together with the transformed coordinates and is accumulated across all coupling blocks when evaluating probabilities and training losses.

By combining coordinate partitioning, equivariant parameter prediction, and expressive spline transformations, the rational quadratic spline coupling layer provides a powerful yet exactly invertible building block for constructing normalizing flows.

In [ ]:
class RQSCouplingBlock(nn.Module):
    """
    Rational-quadratic spline coupling layer.

    The input coordinates are partitioned into two subsets

        x = (x_identity, x_transformed)

    where

    - x_identity is left unchanged,
    - x_identity is used to predict spline parameters,
    - x_transformed is mapped through a monotonic
      rational-quadratic spline.

    The spline parameters are predicted by an equivariant
    transformer network, ensuring permutation equivariance
    with respect to particle exchange.

    Parameters
    ----------
    target_coordinates : list[int]
        Coordinate indices transformed by the coupling layer.

        For example

            [0]

        transforms only x coordinates, while

            [0, 2]

        transforms x and z coordinates.

    n_particles : int
        Number of particles.

    dimensions : int
        Number of spatial dimensions.

    n_bins : int, default=8
        Number of bins in the rational-quadratic spline.

    left, right : float, default=(-1, 1)
        Lower and upper bounds of the spline input domain.

    bottom, top : float, default=(-1, 1)
        Lower and upper bounds of the spline output domain.

    Notes
    -----
    The equivariant network predicts, for each transformed
    coordinate, the spline

        widths,
        heights,
        slopes

    resulting in

        3 * n_bins

    parameters per transformed degree of freedom.

    The final slope is constrained to match the first one,
    making the spline periodic.
    """

    def __init__(
        self,
        target_coordinates,
        n_particles,
        dimensions,
        n_bins=8,
        left=-1.0,
        right=1.0,
        bottom=-1.0,
        top=1.0,
    ):

        super().__init__()

        self.n_particles = n_particles
        self.dimensions = dimensions

        self.n_bins = n_bins

        self.left = left
        self.right = right

        self.bottom = bottom
        self.top = top

        self.identity_dim = (dimensions - len(target_coordinates))
        self.transformed_dim = (len(target_coordinates))

        (identity_indices, transformed_indices) = get_target_indices(target_coordinates, n_particles, dimensions)

        self.identity_indices = identity_indices
        self.transformed_indices = transformed_indices

        self.n_spline_parameters = (3 * self.n_bins)

        # Equivariant network predicting spline parameters
        self.network = ParameterEquivariantNetwork(
            input_size=self.identity_dim,
            output_size=(self.transformed_dim * self.n_spline_parameters),
        )

    def forward(
        self,
        x,
        inverse=False,
    ):
        """
        Apply the coupling transformation.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor with shape

                [batch_size, dofs]

        inverse : bool, default=False
            If True, apply the inverse spline
            transformation.

        Returns
        -------
        y : torch.Tensor
            Transformed tensor with the same shape as x.

        logdetJ : torch.Tensor
            Logarithm of the Jacobian determinant with shape

                [batch_size, 1].
        """

        # --------------------------------------------------
        # Split coordinates
        # --------------------------------------------------

        x_identity = x[:,self.identity_indices]
        x_transformed = x[:,self.transformed_indices]

        # --------------------------------------------------
        # Predict spline parameters
        # --------------------------------------------------
        #
        # Parameters are organized as
        #
        #     [widths | heights | slopes]
        #
        # for each transformed coordinate.
        #
        # --------------------------------------------------

        parameters = self.network(x_identity)

        parameters = rearrange(
            parameters,
            "b (d p) -> b d p",
            d=len(self.transformed_indices),
        )

        widths = parameters[:,:,: self.n_bins]
        heights = parameters[:,:,self.n_bins : 2 * self.n_bins]
        slopes = parameters[:,:,2 * self.n_bins :]

        # Periodic spline:
        # match first and last slope
        slopes = torch.cat([slopes, slopes[..., [0]]], dim=-1)

        # --------------------------------------------------
        # Apply spline transformation
        # --------------------------------------------------

        x_transformed, partial_logdetJ = (
            rational_quadratic_spline(
                x_transformed,
                widths,
                heights,
                slopes,
                inverse=inverse,
                left=self.left,
                right=self.right,
                bottom=self.bottom,
                top=self.top,
                enable_identity_init=True,
            )
        )

        # --------------------------------------------------
        # Reassemble output
        # --------------------------------------------------

        y = x.clone()

        y[:, self.transformed_indices] = x_transformed
        logdetJ = partial_logdetJ.sum(dim=-1, keepdim=True)

        return y, logdetJ

### 2D) Circular Shift

### Circular Shift Layer

The circular shift layer applies the same learnable translation to all coordinates,

$$
x_i \longrightarrow x_i + s,
$$

followed by wrapping back into the periodic domain. The shift vector $s$ is optimized during training.

This layer is often inserted between coupling blocks to improve mixing and allow information to propagate more efficiently through the network. Since translations preserve volume, the transformation is exactly invertible and contributes no Jacobian term,

$$
\log |\det J| = 0.
$$

Although simple, circular shifts can significantly increase the expressivity of a flow when combined with coupling layers.

In [ ]:
class CircularShift(nn.Module):
    """
    Learnable circular translation on a periodic domain.

    The transformation applies the same learnable shift
    to every particle coordinate,

        x -> x + s

    followed by wrapping back into the periodic box.

    Parameters
    ----------
    n_particles : int
        Number of particles.

    dimensions : int
        Spatial dimension.

    box_length : float, default=2
        Periodic box length. Coordinates are assumed
        to lie in

            [-box_length/2, box_length/2].

    Notes
    -----
    The transformation is volume preserving and therefore

        log |det J| = 0.

    The shift vector is learned during training.
    """
    def __init__(self, 
                 n_particles, 
                 dimensions, 
                 box_length=2.
    ):
        super().__init__()

        self.n_particles = n_particles
        self.dimensions = dimensions
        self.dofs = n_particles * dimensions
        self.box_length = box_length

        # Circular shift: this parameter must be learnable
        self.circular_shift = nn.Parameter(torch.zeros(dimensions))

    def forward(
            self, 
            x, 
            inverse=False
    ):
        """
        Learnable circular translation on a periodic domain.

        The transformation applies the same learnable shift
        to every particle coordinate,

            x -> x + s

        followed by wrapping back into the periodic box.

        Parameters
        ----------
        n_particles : int
            Number of particles.

        dimensions : int
            Spatial dimension.

        box_length : float, default=2
            Periodic box length. Coordinates are assumed
            to lie in

                [-box_length/2, box_length/2].

        Notes
        -----
        The transformation is volume preserving and therefore

            log |det J| = 0.

        The shift vector is learned during training.
        """ 
        # Circular shift in "box" [-L/2,L/2]
        x = x.view(-1, self.n_particles, self.dimensions)
        
        sign = -1.0 if inverse else 1.0
        x = x + sign*self.circular_shift        
        x = x - self.box_length*torch.round(x/self.box_length)
        
        x = x.view(-1, self.n_particles*self.dimensions)

        part_log_det = x.new_zeros(x.shape[0], 1)

        return x, part_log_det

### 2E) Lattice Displacement

The Einstein crystal prior is naturally defined in terms of particle displacements from a reference lattice. For this reason, it is often advantageous to express configurations in lattice-displacement coordinates before applying the normalizing flow.

Given a reference configuration $\mathbf{r}^{\,\mathrm{ref}}$ the transformation maps particle positions $\mathbf{r}$ to displacements

$$
\mathbf{u}
=
\mathbf{r}
-
\mathbf{r}^{\,\mathrm{ref}}.
$$

Because the system is periodic, displacements are computed using the minimum-image convention so that each particle remains associated with its nearest lattice site.

Expressing configurations in terms of lattice displacements removes the large static contribution arising from the crystal structure and allows the network to focus on learning the physically relevant fluctuations around the lattice positions.

The inverse transformation simply reconstructs the particle positions by adding back the reference lattice coordinates. Since the transformation is a translation of the coordinate system, it is volume preserving and contributes no Jacobian term,

$$
\log |\det J| = 0.
$$

This transformation is particularly effective for crystalline systems, where particles remain localized around well-defined lattice sites.

In [ ]:
class LatticeDisplacementTransform(nn.Module):
    """
    Convert particle positions into lattice-displacement
    coordinates.

    The forward transformation maps

        r -> u

    where

        u = r - r_ref

    using the minimum-image convention.

    The inverse transformation reconstructs particle
    coordinates from lattice displacements.

    Parameters
    ----------
    reference_configuration : torch.Tensor
        Reference FCC configuration with shape

            [n_particles * dimensions]

    n_particles : int
        Number of particles.

    dimensions : int
        Number of spatial dimensions.

    box_length : float
        Simulation box length.

    Notes
    -----
    This transformation is volume preserving,

        log |det J| = 0.
    """

    def __init__(
        self,
        reference_configuration,
        n_particles,
        dimensions,
        box_length,
    ):

        super().__init__()

        self.n_particles = n_particles
        self.dimensions = dimensions
        self.dofs = n_particles * dimensions

        self.box_length = box_length

        self.register_buffer(
            "reference_configuration",
            reference_configuration.clone(),
        )

    def forward(
        self,
        x,
        inverse=False,
    ):
        """
        Apply the lattice-displacement transform.

        Parameters
        ----------
        x : torch.Tensor
            Tensor with shape

                [batch_size, dofs]

        inverse : bool, default=False
            Whether to apply the inverse transform.

        Returns
        -------
        y : torch.Tensor
            Transformed coordinates.

        logdetJ : torch.Tensor
            Zero tensor with shape

                [batch_size, 1].
        """

        reference = self.reference_configuration.view(
            1,
            self.n_particles,
            self.dimensions,
        )

        positions = x.view(
            -1,
            self.n_particles,
            self.dimensions,
        )

        if inverse:

            positions = positions + reference
            positions -= (self.box_length * torch.round(positions / self.box_length))

        else:

            positions = positions - reference
            positions -= (self.box_length * torch.round(positions / self.box_length))

        y = positions.reshape(
            -1,
            self.dofs,
        )

        logdetJ = y.new_zeros(
            y.shape[0],
            1,
        )

        return y, logdetJ

## 4. Assemble the Flow

The full normalizing flow is constructed by stacking multiple **invertible blocks**. Since the composition of invertible transformations is itself invertible, the resulting model remains exactly reversible regardless of the number of blocks used.

An additional advantage of this construction is that the logarithm of the Jacobian determinant remains easy to evaluate. If the flow is written as a composition of transformations,

$$
F = F_n \circ F_{n-1} \circ \cdots \circ F_1,
$$

then the total log-determinant is simply the sum of the contributions from the individual blocks,

$$
\log |\det J_F|
=
\sum_i
\log |\det J_{F_i}|.
$$

Each block is itself composed of several inner coupling units. An inner coupling unit consists of a circular shift followed by two spline coupling layers acting on complementary coordinate subsets. The first coupling layer transforms the coordinates in target, while the second transforms those in complement. Together, the two layers ensure that all coordinates associated with a given partition are updated before moving to the next partition.

For example, in two dimensions:

```text
Inner unit:
    CircularShift
        │
        ▼
    Transform x | condition on y
        │
        ▼
    Transform y | condition on x
```

As a result, both coordinate directions are modified during a single inner coupling unit.

Similarly, in three dimensions a block contains one inner coupling unit for each coordinate partition returned by get_targets. Together, these units ensure that all coordinate directions are repeatedly transformed and conditioned upon.

This guarantees that information can propagate between all coordinate directions while preserving exact invertibility.

The sequence of coordinate splits used within each block is generated automatically by the helper function `get_targets`, which returns the list of coordinate partitions appropriate for a system of a given dimensionality and for a specified number of blocks.

Increasing the number of blocks increases the expressive power of the flow, allowing it to represent progressively more complex transformations between the prior and posterior distributions. At the same time, exact invertibility and tractable Jacobian evaluation are retained by construction.


An attractive feature of this architecture is its scalability. Each additional block is constructed by repeating the same pattern of coupling units, differing only in the values of their trainable parameters. As a consequence, the total number of trainable parameters grows linearly with the number of blocks,

$$
N_{\mathrm{parameters}}
\propto
N_{\mathrm{blocks}}.
$$

Increasing the depth of the flow therefore provides a straightforward way to increase its expressive power while preserving the modular structure of the architecture. In practice, this allows one to systematically trade computational cost for model flexibility by adjusting the number of blocks without modifying any other component of the network.


In [ ]:
n_blocks = 1

### 4A) Helper functions to split coordinates

In [ ]:
def get_targets(dimensions):
    """
    Generate all coordinate masks and their complements.

    Parameters
    ----------
    dimensions : int
        Number of spatial dimensions.

    Returns
    -------
    list[tuple[tuple[int], tuple[int]]]
        List containing

            (target_coordinates,
             complement_coordinates)

        pairs.

    Examples
    --------
    For dimensions = 3:

        [
            ((0,),   (1, 2)),
            ((1,),   (0, 2)),
            ((2,),   (0, 1)),
            ((0, 1), (2,)),
            ((0, 2), (1,)),
            ((1, 2), (0,))
        ]
    """

    coordinate_indices = tuple(range(dimensions))

    targets = []

    for n_target_coordinates in range(1, dimensions,):

        for target_coordinates in combinations(coordinate_indices, n_target_coordinates,):

            complement_coordinates = tuple(
                coordinate
                for coordinate in coordinate_indices
                if coordinate not in target_coordinates
            )

            targets.append(
                (
                    target_coordinates,
                    complement_coordinates,
                )
            )

    return targets

### 4B) Inner Coupling Units

### Inner Coupling Unit

The basic building block of the flow is the **inner coupling unit**, which consists of a circular shift followed by two rational quadratic spline coupling layers:

```text
Input coordinates
        │
        ▼
Circular shift
        │
        ▼
RQS(target)
        │
        ▼
RQS(complement)
        │
        ▼
Output coordinates
```

The two coupling layers act on complementary subsets of the coordinate directions. For example, in three dimensions, the first coupling layer may transform the $x$ coordinates while conditioning on $y$ and $z$, whereas the second transforms the remaining coordinates conditioned on the updated $x$ coordinates. As a result, all coordinates are modified within a single inner unit.

The circular shift is inserted before the coupling transformations. Since it is invertible and has unit Jacobian, it does not increase the cost of evaluating probabilities. Its role is simply to move information across the periodic coordinate domain and improve communication between successive coupling layers.

This idea was introduced in the Boltzmann Generator framework of Wirnsberger *et al.* and was found to improve the efficiency and expressivity of the resulting flow. In practice, the circular shift acts as a simple mixing operation that complements the more flexible spline transformations.

The coordinate partitions used by the coupling layers are generated automatically by the helper function `get_targets`. For each partition, an inner coupling unit is constructed and the entire sequence is then repeated `n_blocks` times. Since every component is invertible, the resulting architecture remains exactly invertible regardless of its depth.

The final flow is constructed by repeating the sequence of inner coupling units `n_blocks` times. In the implementation, this is achieved by creating deep copies of the units:

```python
coupling_blocks = [
    copy.deepcopy(block)
    for _ in range(n_blocks)
    for block in coupling_unit
]
```

The use of `copy.deepcopy` is important. Without it, the same Python object would be inserted multiple times into the architecture, causing all repetitions of a given unit to share the same trainable parameters. In that case, updating one occurrence during training would automatically update all others, significantly reducing the expressive power of the flow.

By creating deep copies, each repetition receives its own independent set of parameters. Although the architecture of every block is identical, each block is therefore free to learn a different transformation. This allows the flow to progressively refine the transport map as information propagates through successive layers.

### Construct the Inner Coupling Units

<div class="alert alert-block alert-info">

**TASK**

1. Complete the `coupling_unit.extend([...])` section used to construct the architecture of a single inner coupling unit.

2. Add a `CircularShift` layer followed by two `RQSCouplingBlock` layers.

3. Configure the first spline coupling layer to transform the coordinates in `target`.

4. Configure the second spline coupling layer to transform the coordinates in `complement`.

</div>

In [ ]:
targets = get_targets(
    dimensions=dimensions,
)

for target, complement in targets:

    print(
        f"Target: {target}\n"
        f"Complement :{complement}\n"
    )

coupling_unit = []

for target, complement in targets:

    coupling_unit.extend(
        [
            ... # your code goes here
        ]
    )

coupling_blocks = [
    copy.deepcopy(block)
    for _ in range(n_blocks)
    for block in coupling_unit
]

### 4C) Flow assembler with transformation layers

Having defined the prior and posterior systems, the transformation layers, and the sequence of coupling blocks, we can now assemble the complete normalizing flow.

The first step consists of generating a reference lattice configuration,

```python
reference_configuration = system.init_conf()
```

which is used by the lattice-displacement transformations to convert particle positions into displacements relative to their equilibrium lattice sites.

The flow is then created by instantiating the `NormalizingFlow` class. The constructor receives four main ingredients:

* the **prior distribution**,
* the **posterior distribution**,
* the sequence of **invertible coupling blocks**,
* the lists of **prior-side** and **posterior-side transformations**.

The coupling blocks define the trainable transport map in network coordinates, while the transformation layers convert between physical coordinates and the representation used internally by the flow.

In the present example, both the prior and posterior are equipped with a single transformation layer:

```text
Physical coordinates
        │
        ▼
Lattice displacements
        │
        ▼
Coupling blocks
        │
        ▼
Lattice displacements
        │
        ▼
Physical coordinates
```

The prior-side transformation converts configurations sampled from the Einstein crystal into lattice-displacement coordinates before they are processed by the flow. Similarly, the posterior-side transformation converts Lennard-Jones configurations into the same representation when the flow is used in the encoding direction.

Using identical coordinate systems on both sides simplifies the learning problem considerably. Instead of operating directly on absolute particle positions, the flow only needs to model the differences between the displacement distributions of the Einstein crystal and the Lennard-Jones crystal.

Although only a lattice-displacement transformation is used in this workshop, the architecture is fully modular. Additional transformations such as coordinate normalization, symmetry-removal layers, or dimensionality-reduction transformations can be inserted into either transformation pipeline without modifying the coupling blocks themselves.


### Assemble the Normalizing Flow

<div class="alert alert-block alert-info">

**TASK**

1. Instantiate the `NormalizingFlow` class.

2. Assign the Einstein crystal to the `prior` argument and the Lennard-Jones system to the `posterior` argument.

3. Pass the list of coupling blocks constructed in the previous section to the `blocks` argument.

4. Add a `LatticeDisplacementTransform` to both the prior-side and posterior-side transformation pipelines.

</div>

> **Hint**
>
> The reference lattice can be obtained directly from the target system:
>
> ```python
> reference_configuration = system.init_conf()
> ```
>
> This configuration will be used to construct the `LatticeDisplacementTransform` layers on both the prior and posterior sides of the flow. Remember that the prior and posterior systems may have different box lengths, so use the appropriate value when instantiating each transformation.

In [ ]:
# --------------------------------------------------
# Flow
# --------------------------------------------------

flow = NormalizingFlow(

    prior= ... # Your code goes here,
    posterior= ... # Your code goes here,

    blocks= # Your code goes here,

    prior_sided_transformations=[

        ... # Your code goes here
    ],

    posterior_sided_transformations=[

        ...  # Your code goes here
    ],

).to(device)

In [ ]:
# ==================================================
# Flow Summary
# ==================================================

print("=" * 80)
print("FLOW SUMMARY")
print("=" * 80)

print()
print(f"Prior      : {type(flow.prior).__name__}")
print(f"Posterior  : {type(flow.posterior).__name__}")

# --------------------------------------------------
# Transformations
# --------------------------------------------------

print()
print(
    f"Prior-side transformations      : "
    f"{len(flow.prior_sided_transformations)}"
)

for i, transform in enumerate(
    flow.prior_sided_transformations
):
    print(
        f"  [{i}] {type(transform).__name__}"
    )

# --------------------------------------------------
# Flow architecture
# --------------------------------------------------

n_inner_units = len(targets)

print()
print(
    f"Flow blocks                     : "
    f"{n_blocks}"
)

print(
    f"Inner coupling units / block    : "
    f"{n_inner_units}"
)

print(
    f"Total inner coupling units      : "
    f"{n_blocks * n_inner_units}"
)

print()

for i, (target, complement) in enumerate(targets):

    print(
        f"  Unit {i:02d}: "
        f"Shift → "
        f"RQS({target}) → "
        f"RQS({complement})"
    )

# --------------------------------------------------
# Posterior transformations
# --------------------------------------------------

print()
print(
    f"Posterior-side transformations  : "
    f"{len(flow.posterior_sided_transformations)}"
)

for i, transform in enumerate(
    flow.posterior_sided_transformations
):
    print(
        f"  [{i}] {type(transform).__name__}"
    )

# --------------------------------------------------
# Parameters
# --------------------------------------------------

print()

n_parameters = sum(
    p.numel()
    for p in flow.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters            : "
    f"{n_parameters:,}"
)

## 5. Architecture Checks

### 5A) Invertibility

Before training the normalizing flow, it is important to verify that the implementation satisfies the mathematical properties required by the change-of-variables formula. The following cell performs a series of sanity checks on the flow and its transformation layers.

First, we verify that the flow is initialized close to the identity transformation by measuring

$$
\langle |F(z)-z| \rangle,
$$

where $z$ denotes configurations sampled from the prior distribution. Since the spline coupling layers are initialized to represent approximately identity maps, this quantity should be close to zero. You can change this by modifying the `init_weights` method of the class `NormalizingFlow` (check comments therein). Note that the checks we perform next should pass even without identity initialization.

Next, we test invertibility by applying the forward and inverse transformations in succession,

$$
z
\;\xrightarrow{F}\;
x
\;\xrightarrow{F^{-1}}\;
z_{\mathrm{rec}}.
$$

The reconstructed configuration $z_{\mathrm{rec}}$ should coincide with the original configuration $z$ up to numerical precision.

Two reconstruction errors are reported. The **absolute reconstruction error** compares particle coordinates directly, while the **relative reconstruction error** compares particle positions relative to a reference particle. The latter is particularly useful if translational degrees of freedom have been removed, since in that case the flow may reconstruct the correct configuration up to an arbitrary global translation.

Finally, we verify the consistency of the Jacobian determinant. Since the forward and inverse transformations are exact inverses of one another, their log-determinants should satisfy

$$
\log
\left|
\det
\frac{\partial F}{\partial z}
\right|
+
\log
\left|
\det
\frac{\partial F^{-1}}{\partial x}
\right|
=
0.
$$

Any significant deviation from zero indicates an inconsistency in the implementation of either the transformation or its Jacobian.

Passing all of these checks provides strong evidence that the flow has been implemented correctly and is ready to be trained.

### Verify Identity Initialization and Invertibility

<div class="alert alert-block alert-info">

**TASK**

1. Sample a small batch of configurations from the prior distribution.

2. Apply the forward transformation of the flow followed by the inverse transformation.

3. Compute the reconstruction error between the original configurations and the reconstructed configurations.

4. Verify that the logarithms of the forward and inverse Jacobian determinants cancel, i.e.

   $$
   \log |\det J_{z \rightarrow x}|
   +
   \log |\det J_{x \rightarrow z}|
   = 0.
   $$

5. Compute the average displacement between the input configurations and their images under the flow,

   $$
   \langle |F(z)-z| \rangle,
   $$

   and verify that it is close to zero when the network is initialized using the identity initialization.

6. Repeat the test using a non-identity initialization and verify that the identity check fails while the invertibility and Jacobian consistency checks continue to pass.

</div>

In [ ]:
print("=" * 80)
print("INVERTIBILITY CHECK")
print("=" * 80)

# ==================================================
# Reconstruction checks
# ==================================================

... # Your code goes here

# --------------------------------------------------
# Identity mapping check
# --------------------------------------------------

... # Your code goes here

# --------------------------------------------------
# Absolute-coordinate reconstruction
# --------------------------------------------------

... # Your code goes here

# --------------------------------------------------
# Relative-coordinate reconstruction
# --------------------------------------------------

... # Your code goes here

# --------------------------------------------------
# Jacobian consistency
# --------------------------------------------------

... # Your code goes here

# --------------------------------------------------
# Report
# --------------------------------------------------

print(
    f"Mean |F(z)-z|                : "
    f"{identity_error:.3e}"
)

print(
    f"Absolute reconstruction error : "
    f"{absolute_error:.3e}"
)

print(
    f"Relative reconstruction error : "
    f"{relative_error:.3e}"
)

print(
    f"Jacobian mismatch             : "
    f"{jacobian_mismatch:.3e}"
)

tol = 1e-4

print()

if identity_error < tol:

    print(
        "✓ Identity initialization"
    )

else:

    print(
        "✗ Identity initialization"
    )

if absolute_error < tol:

    print(
        "✓ Absolute reconstruction"
    )

else:

    print(
        "✗ Absolute reconstruction "
        "(expected if translations are removed)"
    )

if relative_error < tol:

    print(
        "✓ Relative reconstruction"
    )

else:

    print(
        "✗ Relative reconstruction"
    )

if jacobian_mismatch < tol:

    print(
        "✓ Jacobian consistency"
    )

else:

    print(
        "✗ Jacobian consistency"
    )

### 5B) Permutation Equivariance

A key property of the transformer network used in the coupling layers is **permutation equivariance**. Since the particles are physically indistinguishable, the output of the network should not depend on the arbitrary ordering used to store them in memory.

To verify this property, we generate a batch of particle configurations and evaluate the network on them. We then randomly permute the particle labels and evaluate the network a second time.

If the network is permutation equivariant, permuting the particles before the forward pass should produce exactly the same effect as permuting the outputs after the forward pass. In other words, the following relation should hold:

$$
f(Px) = P f(x),
$$

where $P$ denotes a permutation of the particle labels.

### Verify Permutation Equivariance

<div class="alert alert-block alert-info">

**TASK**

1. Generate a batch of random particle configurations.

2. Apply the parameter-equivariant network to the configurations and store the output.

3. Randomly permute the particle labels in the input configurations.

4. Apply the network to the permuted configurations.

5. Undo the permutation on the network outputs.

6. Compare the original output with the unpermuted output and compute the maximum absolute difference between the two.

7. Verify that the equivariance error is close to machine precision.

</div>

In [ ]:
# ==================================================
# Equivariance test
# ==================================================

batch_size = 16

net = ParameterEquivariantNetwork(
    input_size=dimensions,
    output_size=8,
).to(device)

# --------------------------------------------------
# Random particle configurations
# --------------------------------------------------

x = torch.rand(
    batch_size,
    n_particles * dimensions,
    device=device,
)

# --------------------------------------------------
# Random particle permutation
# --------------------------------------------------

permutation = torch.randperm(
    n_particles,
    device=device,
)

inverse_permutation = torch.argsort(
    permutation,
)

# --------------------------------------------------
# Apply permutation to particles
# --------------------------------------------------

x_particles = x.view(
    batch_size,
    n_particles,
    dimensions,
)

x_permuted = x_particles[:, permutation, :]

x_permuted = x_permuted.reshape(
    batch_size,
    n_particles * dimensions,
)

# --------------------------------------------------
# Network outputs
# --------------------------------------------------

with torch.no_grad():

    y = net(x)

    y_permuted = net(x_permuted)

# --------------------------------------------------
# Undo permutation on output
# --------------------------------------------------

y = y.view(
    batch_size,
    n_particles,
    -1,
)

y_permuted = y_permuted.view(
    batch_size,
    n_particles,
    -1,
)

y_permuted = y_permuted[
    :,
    inverse_permutation,
    :
]

# --------------------------------------------------
# Compare
# --------------------------------------------------

max_error = torch.max(
    torch.abs(
        y - y_permuted
    )
).item()

print(
    f"Max equivariance error : "
    f"{max_error:.3e}\n"
)

Before training the flow, it is useful to inspect how the initialized network acts on configurations sampled from the prior distribution.

The following cell compares three quantities:

1. The lattice displacements of configurations sampled from the Einstein crystal prior.
2. The lattice displacements obtained after passing the same configurations through the initialized flow.
3. The difference between the two.

Since the coupling layers are initialized close to the identity transformation, the generated configurations should initially be very similar to the prior configurations. Consequently, the difference between the two sets of displacements should be small.

This visualization provides an intuitive confirmation of the identity initialization discussed previously. It also serves as a useful baseline for comparison with the trained model, where the flow will progressively deform the prior distribution to match the target Lennard-Jones distribution.


In [ ]:
# ==================================================
# Displacement diagnostics
# ==================================================

# --------------------------------------------------
# Sample prior configurations
# --------------------------------------------------

n_samples = 1000

z = prior.sample(n_samples=n_samples)

with torch.no_grad():

    x, _ = flow(z)

# --------------------------------------------------
# Reference lattice
# --------------------------------------------------

reference = system.init_conf().view(n_particles, dimensions)
reference_np = reference.cpu().numpy()

# --------------------------------------------------
# Reshape configurations
# --------------------------------------------------

zp = z.view(
    n_samples,
    n_particles,
    dimensions,
)

xp = x.view(
    n_samples,
    n_particles,
    dimensions,
)

# --------------------------------------------------
# Prior displacements
# --------------------------------------------------

u_prior = zp - reference
u_prior -= system.box_length * torch.round(u_prior / system.box_length)

# --------------------------------------------------
# Generated displacements
# --------------------------------------------------

u_gen = xp - reference
u_gen -= system.box_length * torch.round(u_gen / system.box_length)

# --------------------------------------------------
# Delta displacement
# --------------------------------------------------

delta = xp - zp
delta -= system.box_length * torch.round(delta / system.box_length)

# --------------------------------------------------
# Flatten for visualization
# --------------------------------------------------

u_prior = u_prior.reshape(-1, 3).cpu().numpy()
u_gen = u_gen.reshape(-1, 3).cpu().numpy()
delta = delta.reshape(-1, 3).cpu().numpy()

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig = plt.figure(figsize=(18, 6))

titles = [
    "Prior displacements",
    "Generated displacements",
    "Delta displacement",
]

clouds = [
    u_prior,
    u_gen,
    delta,
]

for i, (cloud, title) in enumerate(
    zip(clouds, titles)
):

    ax = fig.add_subplot(1, 3, i + 1, projection="3d")

    ax.scatter(
        cloud[:, 0],
        cloud[:, 1],
        cloud[:, 2],
        s=2,
        alpha=0.02,
    )

    ax.set_title(title)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    # Symmetric limits

    extent = np.max(np.abs(cloud))
    extent *= 1.05

    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)

plt.tight_layout()
plt.show()

## 6. Relative Effective Sample Size

A useful way to assess the quality of a normalizing flow is through the **relative effective sample size** (rESS). This quantity measures how broadly distributed the importance weights are and therefore provides an estimate of the overlap between the proposal distribution generated by the flow and the target distribution.

Given a set of logarithmic importance weights,

$$
\log w_i,
$$

we first compute the normalized weights

$$
\bar{w}_i
=
\frac{w_i}
{\sum_j w_j}.
$$

The relative effective sample size is then defined as

$$
\mathrm{rESS}
=
\frac{1}{N}\frac{\left(\sum_jw_j\right)^2}{\sum_jw_j^2}
=
\frac{1}
{N \sum_i \bar{w}_i^2},
$$

where $N$ is the number of samples.

The value of rESS is bounded between 0 and 1:

* $\mathrm{rESS}=1$ corresponds to perfectly uniform weights, indicating excellent overlap between the proposal and target distributions.
* Small values of rESS indicate that only a few samples carry most of the statistical weight, implying poor overlap and inefficient importance sampling.

In the context of this workshop, rESS provides a possible diagnostic for monitoring the quality of the learned transport map. As training progresses, successful learning is generally accompanied by an increase in rESS, reflecting the fact that the flow is generating configurations that more closely resemble the target distribution.

It is important to keep in mind that rESS is a very stringent metric. For high-dimensional systems such as the Lennard-Jones crystal considered here, even a model that generates physically reasonable configurations may still exhibit relatively low rESS values. For this reason, rESS should not be interpreted in isolation.

Additional insight can be obtained by inspecting the distribution of the logarithmic importance weights themselves. Quantities such as the mean, variance, and overall shape of the $\log w$ distribution often provide a more nuanced picture of the overlap between the generated and target distributions and can reveal improvements that are not immediately visible through rESS alone.


### Implement the Relative Effective Sample Size

<div class="alert alert-block alert-info">

**TASK**

1. Complete the function `ress(log_w)`.

2. Test your implementation on different sets of weights for which you know what to expect.

</div>

In [ ]:
# ==================================================
# Relative effective sample size
# ==================================================

def ress(log_w):

    ... # Your code goes here

    return ress

## 7. Build Dataset

The reverse KL loss requires samples from the target distribution and therefore relies on reference configurations generated by Monte Carlo sampling. Before training, these configurations are divided into training, validation, and test sets using a random split.

The same split is applied to both configurations and energies, ensuring that every configuration remains associated with its corresponding potential energy.

The split is also applied to the prior reference configurations. Although the Einstein crystal can be sampled exactly and efficiently, keeping a fixed validation and test set allows us to evaluate the model on the same configurations throughout training. This makes the reported metrics reproducible and ensures that improvements in validation performance are due to changes in the model rather than statistical fluctuations in the sampled data.

During training, however, we adopt a different strategy. Since sampling from the prior is inexpensive, configurations used in the forward KL loss are generated on the fly at every iteration. This effectively provides an unlimited supply of independent training samples and avoids overfitting to a finite set of prior configurations.

As a result, the two directions of training are treated differently:

* **Reverse KL ($x \rightarrow z$):** uses a fixed dataset of configurations sampled from the target distribution.
* **Forward KL ($z \rightarrow x$):** uses freshly generated prior samples during training, but fixed prior samples during validation and testing.

This hybrid approach combines the efficiency of exact prior sampling with the reproducibility required for a reliable evaluation of the model.


In [ ]:
# ==================================================
# Dataset sizes
# ==================================================

n_samples = reference_samples_system.shape[0]

train_fraction = 0.8
validation_fraction = 0.1
test_fraction = 0.1

n_train = int(train_fraction * n_samples)
n_validation = int(validation_fraction * n_samples)
n_test = (n_samples - n_train - n_validation)

print(
    f"Train      : {n_train}"
)
print(
    f"Validation : {n_validation}"
)
print(
    f"Test       : {n_test}"
)

# ==================================================
# Common split indices
# ==================================================

generator = torch.Generator().manual_seed(42)

indices = torch.arange(n_samples)

train_idx, validation_idx, test_idx = random_split(
    indices,
    [n_train, n_validation, n_test],
    generator=generator,
)

train_idx = train_idx.indices
validation_idx = validation_idx.indices
test_idx = test_idx.indices

# ==================================================
# Posterior data
# ==================================================

x_train = reference_samples_system[train_idx]
energy_x_train = reference_energy_system[train_idx]

x_validation = reference_samples_system[validation_idx]
energy_x_validation = reference_energy_system[validation_idx]

x_test = reference_samples_system[test_idx]
energy_x_test = reference_energy_system[test_idx]

# ==================================================
# Prior data
# ==================================================

z_train = reference_samples_prior[train_idx]
energy_z_train = reference_energy_prior[train_idx]

z_validation = reference_samples_prior[validation_idx]
energy_z_validation = reference_energy_prior[validation_idx]

z_test = reference_samples_prior[test_idx]
energy_z_test = reference_energy_prior[test_idx]

print()
print(
    f"x_train shape        : {x_train.shape}"
)
print(
    f"x_validation shape   : {x_validation.shape}"
)
print(
    f"x_test shape         : {x_test.shape}"
)

print()
print(
    f"z_train shape        : {z_train.shape}"
)
print(
    f"z_validation shape   : {z_validation.shape}"
)
print(
    f"z_test shape         : {z_test.shape}"
)

## 8. Training

Once the datasets have been prepared, training the normalizing flow follows a fairly standard optimization procedure. At each iteration, a batch of configurations is used to evaluate the chosen loss function, gradients are computed by backpropagation, and the network parameters are updated using an optimizer such as Adam.

The main difference with respect to a conventional supervised learning problem is that the flow can be trained in two complementary directions.

The **reverse KL loss** (or maximum likelihood objective) uses configurations sampled from the target distribution. A batch of target configurations is mapped to the latent space and the likelihood assigned by the flow is maximized. This direction therefore requires reference data obtained from Monte Carlo sampling.

The **forward KL loss** (or Boltzmann Generator objective) operates in the opposite direction. Configurations are sampled from the prior, transformed by the flow, and evaluated using the target potential energy. Since prior samples can be generated exactly and efficiently, this loss can be evaluated without requiring additional reference data.

The total training objective is a weighted combination of the two losses,

$$
\mathcal{L}
=
w_{xz}\mathcal{L}_{xz}
+
w_{zx}\mathcal{L}_{zx},
$$

where the coefficients $w_{xz}$ and $w_{zx}$ determine whether the reverse KL, the forward KL, or both objectives are used during training.

In addition to the training loss, we monitor several validation metrics, including the validation losses and the relative effective sample size. A learning-rate scheduler is used to reduce the learning rate when progress stalls, and early stopping is employed to terminate training once the validation metric has ceased to improve for a specified number of epochs.

Apart from the presence of the two training directions, the overall training loop is therefore very similar to that used in many deep-learning applications: batches are processed sequentially, gradients are accumulated through automatic differentiation, and the model parameters are updated iteratively until convergence.


> **Note**
>
> As a starting point, we train the normalizing flow using **both** the forward and reverse KL objectives. This is often a good default choice, as the two losses provide complementary information:
>
> - The **reverse KL** (maximum likelihood) loss encourages the flow to reproduce the target distribution in regions sampled by the reference data.
> - The **forward KL** (Boltzmann Generator) loss encourages the flow to generate physically relevant configurations when sampling from the prior.
>
> Combining the two objectives typically leads to a more stable training procedure and can improve both sample quality and overlap with the target distribution.

In [ ]:
# ==================================================
# Training parameters
# ==================================================

n_epochs = 100

batch_size = 512

w_xz = 1.0
w_zx = 1.0

learning_rate = 1e-4
plateau_threshold = 1e-2

early_stopping_patience = 15
early_stopping_threshold = 1e-2

# ==================================================
# Optimizer and scheduler
# ==================================================

optimizer = torch.optim.Adam(
    [
        parameter 
        for parameter in flow.parameters() 
        if parameter.requires_grad
    ],
    lr=learning_rate,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    threshold=plateau_threshold,
    threshold_mode="rel",
)

### Complete the Training and Validation Loops

<div class="alert alert-block alert-info">

**TASK**

1. Complete the training and validation routines by computing both the forward KL and reverse KL losses.

2. During training, evaluate the reverse KL loss using batches of reference Lennard-Jones configurations from the training set.

3. During training, generate fresh Einstein crystal configurations on the fly and use them to evaluate the forward KL loss.

4. Combine the two loss contributions using the corresponding weights defined in the training script and perform the optimizer update.

5. For the validation forward KL loss, do **not** generate new prior configurations. Instead, use the fixed validation prior dataset created during the dataset preparation stage.

</div>

In [ ]:
# ==================================================
# Early stopping
# ==================================================

best_metric = np.inf
best_state = copy.deepcopy(flow.state_dict())

patience_counter = 0

# ==================================================
# Training history
# ==================================================

history = []

# ==================================================
# Training loop
# ==================================================
try:
    for epoch in range(1, n_epochs + 1):

        flow.train()

        permutation = torch.randperm(x_train.shape[0], device=x_train.device)
        n_batches = (x_train.shape[0] // batch_size)

        progress_bar = tqdm(
            range(n_batches),
            leave=False,
            desc=f"Epoch {epoch:4d}",
        )

        for batch in progress_bar:

            indices = permutation[batch * batch_size:(batch + 1) * batch_size]

            loss_xz = torch.tensor(0.0, device=flow.device)
            loss_zx = torch.tensor(0.0, device=flow.device)

            # ------------------------------------------
            # Reverse KL / Maximum likelihood
            # ------------------------------------------

            if w_xz > 0:

                ... # Your code goes here

            # ------------------------------------------
            # Forward KL / Boltzmann generator
            # ------------------------------------------

            if w_zx > 0:

                ... # Your code goes here

            # ------------------------------------------
            # Total loss
            # ------------------------------------------

            loss = w_xz * loss_xz + w_zx * loss_zx

            optimizer.zero_grad()
            
            loss.backward()
            optimizer.step()

            progress_bar.set_postfix(
                loss_xz=f"{loss_xz.item():.3f}",
                loss_zx=f"{loss_zx.item():.3f}",
            )

        # ==================================================
        # Validation
        # ==================================================

        flow.eval()

        with torch.no_grad():

            # ------------------------------------------
            # x -> z
            # ------------------------------------------

                ... # Your code goes here

            ress_xz = ress(logw_xz).item()

            # ------------------------------------------
            # z -> x
            # ------------------------------------------

                ... # Your code goes here

            ress_zx = ress(logw_zx).item()

        # ----------------------------------------------
        # Validation metric
        # ----------------------------------------------

        validation_metric = (1/(w_xz + w_zx) * (w_xz*val_loss_xz + w_zx*val_loss_zx)).item()
        scheduler.step(validation_metric)

        # ----------------------------------------------
        # Early stopping
        # ----------------------------------------------

        if validation_metric < best_metric * (1 - early_stopping_threshold):

            best_metric = validation_metric

            best_state = copy.deepcopy(flow.state_dict())

            patience_counter = 0

        else:

            patience_counter += 1

        # ----------------------------------------------
        # Logging
        # ----------------------------------------------

        current_lr = optimizer.param_groups[0]["lr"]

        history.append(
            [
                epoch,

                loss_xz.item(),
                loss_zx.item(),

                val_loss_xz.item(),
                val_loss_zx.item(),

                ress_xz,
                ress_zx,

                validation_metric,

                current_lr,
            ]
        )

        print(

            f"Epoch {epoch:4d} | "
            f"train_loss_xz = {loss_xz:.3f} | "
            f"train_loss_zx = {loss_zx:.3f} | "
            f"eval_loss_xz = {val_loss_xz:.3f} | "
            f"eval_loss_zx = {val_loss_zx:.3f} | "
            f"rESS_xz = {ress_xz:.3f} | "
            f"rESS_zx = {ress_zx:.3f} | "
            f"loss = {validation_metric:.3f} | "
            f"lr = {current_lr:.2e}"
        )

        # ----------------------------------------------
        # Stop if plateau persists
        # ----------------------------------------------

        if patience_counter >= early_stopping_patience:

            print()

            print(
                f"Early stopping after "
                f"{epoch} epochs."
            )

            break

except KeyboardInterrupt:

    print()
    print(
        f"Training interrupted at "
        f"epoch {epoch}."
    )

finally:

    # ==================================================
    # Restore best model
    # ==================================================

    flow.load_state_dict(best_state)

    history = np.asarray(history)

    print()
    print(
        f"Best validation = "
        f"{best_metric:.4f}"
    )

In [ ]:
# ==================================================
# Training history
# ==================================================

epochs = history[:, 0]

train_loss_xz = history[:, 1]
train_loss_zx = history[:, 2]

val_loss_xz = history[:, 3]
val_loss_zx = history[:, 4]

ress_xz = history[:, 5]
ress_zx = history[:, 6]

learning_rate = history[:, 8]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

# --------------------------------------------------
# Reverse KL (x -> z)
# --------------------------------------------------

axes[0].plot(
    epochs,
    train_loss_xz,
    label="Train",
)

axes[0].plot(
    epochs,
    val_loss_xz,
    label="Validation",
)

axes[0].set_title(
    "Reverse KL (x → z)"
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[0].legend()

# --------------------------------------------------
# Forward KL (z -> x)
# --------------------------------------------------

axes[1].plot(
    epochs,
    train_loss_zx,
    label="Train",
)

axes[1].plot(
    epochs,
    val_loss_zx,
    label="Validation",
)

axes[1].set_title(
    "Forward KL (z → x)"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Generation

### Generating Samples and Computing Importance Weights

After training, the quality of the learned transport map can be evaluated by generating configurations from the flow and comparing them to the target distribution.

To do so, we first sample latent configurations

$$
z \sim \mu_Z(z),
$$

from the Einstein crystal prior and transform them through the learned flow,

$$
x = F(z).
$$

The resulting configurations constitute the model's approximation of the Lennard-Jones distribution. We then evaluate their potential energy and compute the corresponding logarithmic importance weights,

$$
\log w
= -\beta U(F(z))
+
\log \mu_Z(z)
-
\log
\left|
\det
J(F(z))
\right|.
$$

From these weights we compute the relative effective sample size, which provides a quantitative measure of the overlap between the learned distribution and the target distribution.

### Generate Configurations with the Trained Flow

<div class="alert alert-block alert-info">

**TASK**

1. Use the flow to generate `n_generated_samples` new configurations for the Lennard-Jones crystal starting from Einstein crystal configurations

2. Compute the Lennard-Jones energies of the generated configurations.

3. Compute the logarithmic importance weights associated with the generated samples.

4. Use these weights to evaluate the relative effective sample size (rESS) of the trained model.

</div>

In [ ]:
n_generated_samples = 10000

with torch.no_grad():
    
    flow.eval()

    ... # Your code goes here

### 9A) Comparison with identity using weights

For comparison, we also evaluate the importance weights associated with the **identity transformation**,

$$
x = z.
$$

In this case, the generated configurations are simply samples from the Einstein crystal prior evaluated in the Lennard-Jones potential. The corresponding weights and rESS therefore provides a useful baseline that quantifies how well the prior alone approximates the target distribution.

Comparing the rESS of the trained flow with that of the identity transformation allows us to assess how much the learned transport map has improved the overlap between the two distributions. Throughout the remainder of the notebook, the identity transformation will serve as a convenient reference when evaluating the performance of the model.


### Compare the Importance Weights with the Identity Transformation

<div class="alert alert-block alert-info">

**TASK**

1. Computes the weights and the rESS associated with the identity transformation.

2. Use the two sets of weights to compute the corresponding relative effective sample sizes.

3. Use the plotting scripts below to compare the distributions of the logarithmic weights obtained from the identity transformation and from the trained flow.

</div>

In [ ]:
... # Your code goes here

We now print some metrics and plot the distributions

In [ ]:
flow_log_w = log_w - torch.max(log_w)
id_log_w_shifted = id_log_w - torch.max(id_log_w)

print()
print("=" * 60)
print("IMPORTANCE SAMPLING DIAGNOSTICS")
print("=" * 60)

print(
    f"{'Metric':20s}"
    f"{'Identity':>15s}"
    f"{'Flow':>15s}"
)

print(
    f"{'rESS':20s}"
    f"{id_ress_zx.item():15.3e}"
    f"{ress_zx.item():15.3e}"
)

print(
    f"{'mean(log w)':20s}"
    f"{id_log_w_shifted.mean().item():15.3f}"
    f"{flow_log_w.mean().item():15.3f}"
)

print(
    f"{'std(log w)':20s}"
    f"{id_log_w_shifted.std().item():15.3f}"
    f"{flow_log_w.std().item():15.3f}"
)

print("=" * 60)

The relative effective sample size provides a compact summary of the quality of the importance weights, but it does not reveal *why* a particular value is obtained. To gain further insight, it is useful to inspect the distribution of the weights themselves.

The logarithmic importance weights are first normalized by subtracting their maximum value,

$$
\log \bar{\omega}_i
=\log \omega_i
-
\max_j
\log \omega_j.
$$

This operation leaves all relative weight ratios unchanged while improving numerical stability and making the distributions easier to compare visually.

In the plot below, the left panel shows the normalized log-weights sorted from smallest to largest. A perfectly uniform set of weights would appear as a horizontal line. In practice, the spread of the curve provides a direct indication of how unevenly the statistical weight is distributed among the generated configurations. Large negative values correspond to configurations that contribute very little to ensemble averages, whereas values close to zero correspond to the most important samples.

The right panel displays the corresponding probability density of the normalized log-weights. This representation summarizes the overall shape of the weight distribution and makes it easier to compare different models.

In both panels, the trained flow is compared against the identity transformation. Since the identity transformation corresponds to using the Einstein crystal directly as a proposal distribution, it provides a useful baseline for assessing the improvement achieved by the learned transport map.

When interpreting these plots, it is often useful to focus on simple statistical descriptors such as the mean and variance of the normalized log-weights. Broad distributions with large negative tails generally indicate poor overlap between the proposal and target distributions, while narrower distributions are typically associated with larger effective sample sizes and more efficient importance sampling.


In [ ]:
# ==================================================
# Relative weight diagnostics
# ==================================================

# --------------------------------------------------
# Normalized log weights
# --------------------------------------------------

log_w_plot = (
    log_w
    - log_w.max()
)

log_w_id_plot = (
    id_log_w
    - id_log_w.max()
)

log_w_plot = (
    torch.sort(log_w_plot)[0]
    .cpu()
    .numpy()
)

log_w_id_plot = (
    torch.sort(log_w_id_plot)[0]
    .cpu()
    .numpy()
)

# --------------------------------------------------
# Figure
# --------------------------------------------------

fig_size = (
    22 * 0.393701,
    11 * 0.393701,
)

fig, ax = plt.subplots(
    1,
    2,
    figsize=fig_size,
    dpi=400,
    sharey=True,
    tight_layout=True,
)

# --------------------------------------------------
# Scientific notation on x-axis
# --------------------------------------------------

formatter = ticker.ScalarFormatter(
    useMathText=True
)

formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))

ax[0].xaxis.set_major_formatter(
    formatter
)

# --------------------------------------------------
# Sorted log weights
# --------------------------------------------------

ax[0].plot(
    log_w_plot,
    lw=1,
    color="C0",
    label=(
        rf"Flow "
        rf"(rESS={ress_zx:.3f})"
    ),
)

ax[0].plot(
    log_w_id_plot,
    alpha=1,
    color="C1",
    label=(
        rf"Identity "
        rf"(rESS={id_ress_zx:.3f})"
    ),
)

ax[0].set_xlabel(
    r"sorted sample $i$"
)

ax[0].set_ylabel(
    r"$\log \bar{\omega}(x^i)$"
)

ax[0].legend(
    frameon=False
)

# --------------------------------------------------
# Side histogram
# --------------------------------------------------

hist, bins = np.histogram(
    log_w_plot,
    bins=100,
    density=True,
)

centers = (
    bins[1:]
    + bins[:-1]
) / 2

hist_id, bins_id = np.histogram(
    log_w_id_plot,
    bins=100,
    density=True,
)

centers_id = (
    bins_id[1:]
    + bins_id[:-1]
) / 2

ax[1].plot(
    hist,
    centers,
    color="C0",
    label="Flow",
)

ax[1].plot(
    hist_id,
    centers_id,
    color="C1",
    label="Identity",
)

ax[1].set_xlabel(
    r"$P(\log \bar{\omega})$"
)

ax[1].legend(
    frameon=False
)

plt.show()

### 9B) Analysis of generated Configurations

Having evaluated the flow using importance weights and rESS, we now turn to a more physically meaningful assessment of the generated configurations.

The first quantities we consider are the **potential energy distribution** and the **radial distribution function** $g(r)$. These provide complementary information about the quality of the generated samples.

The energy distribution probes whether the flow correctly reproduces the thermodynamic properties of the target ensemble. A successful model should generate configurations whose energies follow the same distribution as the reference Monte Carlo samples. In addition to the raw energy distribution generated by the flow, we also compute a **reweighted energy distribution** using the importance weights associated with the generated configurations. In the limit of perfect sampling, this reweighted histogram provides an estimate of the target energy distribution and should therefore coincide with the reference Monte Carlo result. Comparing the raw and reweighted distributions offers valuable insight into the quality of the learned transport map: differences between the two indicate regions where importance sampling corrections remain necessary, while agreement suggests that the flow itself is already generating configurations close to the target ensemble.


The radial distribution function $g(r)$ characterizes the local structure of the system by measuring the probability of finding a particle at a distance $r$ from another particle relative to an ideal gas at the same density. Because it contains information about the shell structure and long-range ordering of the crystal, it provides a sensitive test of whether the generated configurations reproduce the correct spatial correlations.

After examining these ensemble-level observables, we inspect the configurations more directly. Since the flow operates in lattice-displacement coordinates, it is natural to first compare the displacement distributions of the generated and reference configurations. This representation highlights how the flow has learned to deform the Einstein crystal fluctuations into those of the Lennard-Jones crystal.

Finally, we transform the configurations back to Cartesian coordinates and visualize representative particle arrangements. While energy distributions and radial distribution functions provide quantitative measures of agreement, direct inspection of the generated structures offers an intuitive picture of the quality of the learned transport map and can reveal defects or artifacts that may not be immediately visible in aggregate statistics.

Taken together, these analyses provide a progressively more detailed view of the model performance, moving from global thermodynamic observables to local structural information and finally to individual microscopic configurations.

### Analyze the Physical Quality of the Generated Configurations

<div class="alert alert-block alert-info">

**TASK**

1. Complete the implementation of the reweighted energy histogram by using the importance weights computed from the trained flow.

2. Compare the following energy distributions:
   - the reference Lennard-Jones distribution,
   - the prior configurations in the target energy (identity transformation),
   - the generated configurations,
   - the reweighted generated configurations.

3. Discuss the results for importance reweighting.

4. Compare the radial distribution functions of the Einstein crystal, the generated configurations, and the reference Lennard-Jones crystal.

5. Compare the displacement distributions before and after training. In particular, contrast the configurations generated by the initialized network with those generated by the trained model.

6. Discuss how the flow modifies the fluctuations around the lattice sites and whether the resulting displacement patterns resemble those observed in the reference Lennard-Jones configurations.

7. Finally, visualize representative particle configurations in Cartesian coordinates and compare them to the reference samples. Look for possible defects, distortions, or other artifacts that may indicate shortcomings of the learned transport map.

</div>

In [ ]:
# --------------------------------------------------
# RDFs
# --------------------------------------------------

r_prior, g_prior = radial_distribution_function(
    configurations=reference_samples_prior,
    system=prior,
)

r_generated, g_generated = radial_distribution_function(
    configurations=generated_samples,
    system=system,
)

r_system, g_system = radial_distribution_function(
    configurations=reference_samples_system,
    system=system,
)

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4),
)

# --------------------------------------------------
# Energy distributions
# --------------------------------------------------

axes[0].hist(
    reference_energy_system.cpu().numpy(),
    bins=50,
    density=True,
    alpha=0.6,
    label="Reference",
)

axes[0].hist(
    identity_energy.cpu().numpy(),
    bins=50,
    density=True,
    alpha=0.6,
    label="Identity",
)

axes[0].hist(
    generated_energy.cpu().numpy(),
    bins=50,
    density=True,
    alpha=0.6,
    label="Generated",
)

# --------------------------------------------------
# Reweighted generated distribution
# --------------------------------------------------

weights = torch.exp(
    log_w - torch.max(log_w)
)

weights = (
    weights
    / torch.sum(weights)
)

... # Your code goes here

axes[0].set_ylim(None, 0.06)
axes[0].set_xlabel("Energy")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Energy distributions")
axes[0].legend()

# --------------------------------------------------
# Radial distribution functions
# --------------------------------------------------

axes[1].plot(
    r_system,
    g_system,
    label="Reference Lennard-Jones",
)

axes[1].plot(
    r_prior,
    g_prior,
    label="Einstein Crystal",
)

axes[1].plot(
    r_generated,
    g_generated,
    label="Generated Lennard-Jones",
)

axes[1].set_xlabel(r"$r$")
axes[1].set_ylabel(r"$g(r)$")
axes[1].set_title("Radial distribution function")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# Displacement diagnostics
# ==================================================

# --------------------------------------------------
# Sample prior configurations
# --------------------------------------------------

n_samples = 1000

z = prior.sample(n_samples=n_samples)

with torch.no_grad():

    x, _ = flow(z)

# --------------------------------------------------
# Reference lattice
# --------------------------------------------------

reference = system.init_conf().view(n_particles, dimensions)
reference_np = reference.cpu().numpy()

# --------------------------------------------------
# Reshape configurations
# --------------------------------------------------

zp = z.view(
    n_samples,
    n_particles,
    dimensions,
)

xp = x.view(
    n_samples,
    n_particles,
    dimensions,
)

# --------------------------------------------------
# Prior displacements
# --------------------------------------------------

u_prior = zp - reference
u_prior -= system.box_length * torch.round(u_prior / system.box_length)

# --------------------------------------------------
# Generated displacements
# --------------------------------------------------

u_gen = xp - reference
u_gen -= system.box_length * torch.round(u_gen / system.box_length)

# --------------------------------------------------
# Delta displacement
# --------------------------------------------------

delta = xp - zp
delta -= system.box_length * torch.round(delta / system.box_length)

# --------------------------------------------------
# Flatten for visualization
# --------------------------------------------------

u_prior = u_prior.reshape(-1, 3).cpu().numpy()
u_gen = u_gen.reshape(-1, 3).cpu().numpy()
delta = delta.reshape(-1, 3).cpu().numpy()

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig = plt.figure(figsize=(18, 6))

titles = [
    "Prior displacements",
    "Generated displacements",
    "Delta displacement",
]

clouds = [
    u_prior,
    u_gen,
    delta,
]

for i, (cloud, title) in enumerate(
    zip(clouds, titles)
):

    ax = fig.add_subplot(1, 3, i + 1, projection="3d")

    ax.scatter(
        cloud[:, 0],
        cloud[:, 1],
        cloud[:, 2],
        s=2,
        alpha=0.02,
    )

    ax.set_title(title)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    # Symmetric limits

    extent = np.max(np.abs(cloud))
    extent *= 1.05

    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)

plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------
# Reference FCC lattice
# --------------------------------------------------

reference_positions = system.init_conf(as_numpy=True)
reference_positions_batch = (reference_positions[None])

# --------------------------------------------------
# Utility
# --------------------------------------------------

def crystal_cloud(samples, system, n_display):

    indices = np.random.choice(
        samples.shape[0],
        size=min(n_display, samples.shape[0]),
        replace=False,
    )

    positions = (samples[indices].view(-1, n_particles, 3).cpu().numpy())

    # Minimum-image displacements relative to FCC sites

    displacements = (positions - reference_positions_batch)
    displacements -= (system.box_size * np.round(displacements / system.box_size))

    positions = (reference_positions_batch + displacements)

    return positions.reshape(-1, 3)

# --------------------------------------------------
# Build clouds
# --------------------------------------------------

n_display = 500

prior_cloud = crystal_cloud(
    x,
    prior,
    n_display,
)

posterior_cloud = crystal_cloud(
    reference_samples_system,
    system,
    n_display,
)

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig = plt.figure(figsize=(8, 8))

ax = fig.add_subplot(111, projection="3d")

# Prior cloud

prior_handle = ax.scatter(
    prior_cloud[:,0],
    prior_cloud[:,1],
    prior_cloud[:,2],
    s=2,
    alpha=0.03,
)

# Posterior cloud

posterior_handle = ax.scatter(
    posterior_cloud[:,0],
    posterior_cloud[:,1],
    posterior_cloud[:,2],
    s=2,
    alpha=0.03,
)

# FCC lattice sites

fcc_handle = ax.scatter(
    reference_positions[:,0],
    reference_positions[:,1],
    reference_positions[:,2],
    s=10,
    c="k",
    marker="x",
)

legend = ax.legend(
    [prior_handle, posterior_handle, fcc_handle],
    ["Generated Lennard-Jones crystal", "Reference Lennard-Jones crystal", "FCC sites"],
)

for h in legend.legend_handles:
    h.set_alpha(1.0)
    h.set_sizes([40])

half_box = 0.5 * system.box_size

ax.set_xlim(-half_box, half_box)
ax.set_ylim(-half_box, half_box)
ax.set_zlim(-half_box, half_box)

ticks = [-half_box, 0.0, half_box]

ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_zticks(ticks)

ax.set_xticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_yticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_zticklabels(
    [r"$-L/2$", r"$0$", r"$L/2$"]
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax.set_box_aspect((1, 1, 1))

ax.set_title(
    f"{n_display} configurations"
)

# plt.tight_layout()
plt.show()

<div class="alert alert-block alert-success">

**OPTIONAL TASKS**

1. **Train using the forward KL loss only**

    Repeat the training procedure by setting

    ```python
    w_xz = 0
    ```

    so that the model is trained exclusively through the forward KL (energy-based) objective.

    Compare the resulting model with the one obtained using bidirectional training. In particular:

    - Compare the energy distributions, radial distribution functions, and displacement distributions.

    - Compare the logarithmic importance weights and the relative effective sample size.

    - Inspect the generated configurations visually.

    Discuss any issues that arise. For example, does the model appear to focus on a subset of configurations while neglecting others? Can you identify signatures of mode collapse or insufficient exploration of configuration space?

    Finally, think about possible remedies, such as modifying the loss function, changing the prior, introducing symmetry-removal transformations, or combining the forward and reverse objectives.

2. **Modify the prior and target distributions**

    Repeat the experiment using different thermodynamic parameters for the Einstein crystal and/or the Lennard-Jones system.

    For example, you may change:

    - the Einstein crystal width parameter,

    - the temperature,

    - the density.

    Investigate how the overlap between prior and target distributions changes and quantify the effect on the relative effective sample size.


3. **Increase the model capacity**

    If sufficient memory is available, experiment with larger architectures by increasing:

    - the number of flow blocks,

    - the number of spline bins,

    - the transformer depth and hidden dimension.

    Analyze how the increased expressive power affects:

    - training and validation losses,

    - relative effective sample size,

    - energy distributions,

    - radial distribution functions,

    - generated configurations.

    Is the performance limited primarily by model capacity, or by the difficulty of the sampling problem itself?

</div>

## References

[Wirnsberger2022]
P. Wirnsberger, G. Papamakarios, B. Ibarz, S. Racanière, A. J. Ballard, A. Pritzel, and C. Blundell,  
*Normalizing flows for atomic solids*,  
Machine Learning: Science and Technology **3**, 025009 (2022).  
DOI: 10.1088/2632-2153/ac6b16.
